In [3]:
# ============================================================
# CELL 1 — ENVIRONMENT, GLOBAL CONFIG, SEED AND GPU CHECK
# DenseNet121 Combined Eye ROI Deepfake Classification
# ============================================================

import os
import sys
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Single Source of Truth configuration
# Edit only these values when your Drive layout changes.
# ------------------------------------------------------------
CONFIG = {
    "seed": 42,
    "team_member_folder": "Kader",
    "experiment_folder": "Deney 1",
    "roi_folder_candidates": ["Göz", "Goz", "Eye"],
    "model_name": "DenseNet121",
    "roi_name": "eye",
    "image_height": 224,
    "image_width": 224,
    "batch_size": 16,
    "frozen_learning_rate": 1e-3,
    "finetune_learning_rate": 1e-5,
    "dropout_rate": 0.40,
    "frozen_epochs": 12,
    "finetune_epochs": 20,
    "early_stopping_patience": 5,
    "resume_run_id": None,  # e.g. "20260808_1412_eye_densenet121_seed42"
}

SEED = int(CONFIG["seed"])
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 78)
print("DENSENET121 COMBINED EYE ROI DEEPFAKE EXPERIMENT")
print("=" * 78)
print("Python version     :", sys.version.split()[0])
print("TensorFlow version :", tf.__version__)
print("NumPy version      :", np.__version__)
print("Random seed        :", SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("\nGPU detected:")
    for gpu in gpus:
        print(" -", gpu)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth  : ENABLED")
    except RuntimeError as error:
        print("GPU configuration warning:", error)
else:
    print("\nWARNING: GPU could not be detected.")
    print("Colab: Runtime > Change runtime type > T4 GPU")

device_name = tf.test.gpu_device_name()
print("\nTensorFlow device  :", device_name if device_name else "CPU")
print("=" * 78)
print("CELL 1 COMPLETED")


DENSENET121 COMBINED EYE ROI DEEPFAKE EXPERIMENT
Python version     : 3.12.13
TensorFlow version : 2.20.0
NumPy version      : 2.0.2
Random seed        : 42

GPU detected:
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU memory growth  : ENABLED

TensorFlow device  : /device:GPU:0
CELL 1 COMPLETED


In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

print("Exists:", root.exists())

items = list(root.iterdir())

print("Item count:", len(items))

for item in items[:50]:
    print(item)

Exists: True
Item count: 1
/content/drive/MyDrive/AISC DeepFake Çalışmaları


In [7]:
# ============================================================
# CELL 2 — GOOGLE DRIVE + EXACT EYE EXPERIMENT PATHS
# KADER / DENEY 1 / GÖZ
# ============================================================

from google.colab import drive
from pathlib import Path


# ============================================================
# 1. GOOGLE DRIVE MOUNT
# ============================================================

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"


if not MY_DRIVE.exists():
    print("Google Drive is not mounted. Mounting...")
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")


# Final mount validation
if not MY_DRIVE.exists():
    raise RuntimeError(
        "\nGoogle Drive mount failed.\n"
        f"Expected directory does not exist:\n{MY_DRIVE}"
    )


# ============================================================
# 2. EXACT PROJECT DIRECTORY STRUCTURE
# ============================================================

# /content/drive/MyDrive/AISC DeepFake Çalışmaları
AISC_ROOT = (
    MY_DRIVE
    / "AISC DeepFake Çalışmaları"
)

# /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler
EXPERIMENTS_ROOT = (
    AISC_ROOT
    / "Deneyler"
)

# /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader
USER_ROOT = (
    EXPERIMENTS_ROOT
    / "Kader"
)

# /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
EXPERIMENT_ROOT = (
    USER_ROOT
    / "Deney 1"
)

# /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz
EYE_ROOT = (
    EXPERIMENT_ROOT
    / "Göz"
)

# /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
EYE_ROI_ROOT = (
    EYE_ROOT
    / "eye_roi_output"
)

# /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar
RESULTS_ROOT = (
    EXPERIMENT_ROOT
    / "Sonuçlar"
)


# ============================================================
# 3. REQUIRED INPUT DIRECTORY VALIDATION
# ============================================================

REQUIRED_INPUT_DIRS = {
    "MY_DRIVE": MY_DRIVE,
    "AISC_ROOT": AISC_ROOT,
    "EXPERIMENTS_ROOT": EXPERIMENTS_ROOT,
    "USER_ROOT": USER_ROOT,
    "EXPERIMENT_ROOT": EXPERIMENT_ROOT,
    "EYE_ROOT": EYE_ROOT,
    "EYE_ROI_ROOT": EYE_ROI_ROOT,
}


for name, path in REQUIRED_INPUT_DIRS.items():

    if not path.exists():
        raise FileNotFoundError(
            "\nRequired project directory was not found.\n"
            f"Variable : {name}\n"
            f"Path     : {path}\n\n"
            "Check the Google Drive folder structure before continuing."
        )

    if not path.is_dir():
        raise NotADirectoryError(
            "\nExpected a directory but found something else.\n"
            f"Variable : {name}\n"
            f"Path     : {path}"
        )


# ============================================================
# 4. RESULTS DIRECTORY
# ============================================================

# Sonuçlar is an output directory.
# If it somehow does not exist, creating it is safe.

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if not RESULTS_ROOT.exists():
    raise RuntimeError(
        f"Results directory could not be created: {RESULTS_ROOT}"
    )


# ============================================================
# 5. OPTIONAL CONFIG CONSISTENCY CHECK
# ============================================================

# If CONFIG was already created in CELL 1, verify that its
# directory-related values agree with the actual experiment.

if "CONFIG" in globals():

    expected_config_values = {
        "team_member_folder": "Kader",
        "experiment_folder": "Deney 1",
    }

    for key, expected_value in expected_config_values.items():

        if key in CONFIG:

            actual_value = str(CONFIG[key]).strip()

            if actual_value != expected_value:
                raise ValueError(
                    "\nCONFIG directory setting does not match "
                    "the real Google Drive structure.\n"
                    f"CONFIG key : {key}\n"
                    f"Expected   : {expected_value}\n"
                    f"Received   : {actual_value}"
                )


# ============================================================
# 6. EYE ROI DIRECTORY CONTENT CHECK
# ============================================================

try:
    eye_roi_items = list(EYE_ROI_ROOT.iterdir())

except Exception as exc:
    raise RuntimeError(
        "\nEye ROI directory exists but cannot be read.\n"
        f"Path: {EYE_ROI_ROOT}"
    ) from exc


if len(eye_roi_items) == 0:
    raise RuntimeError(
        "\nEye ROI directory is empty.\n"
        f"Path: {EYE_ROI_ROOT}\n"
        "Model training cannot continue without ROI data."
    )


# ============================================================
# 7. BASIC ROI STRUCTURE AUDIT
# ============================================================

eye_roi_dirs = sorted(
    [
        item
        for item in eye_roi_items
        if item.is_dir()
    ],
    key=lambda p: p.name.casefold(),
)

eye_roi_files = sorted(
    [
        item
        for item in eye_roi_items
        if item.is_file()
    ],
    key=lambda p: p.name.casefold(),
)


# ============================================================
# 8. SEARCH FOR METADATA FILES
# ============================================================

# Only search inside the known eye experiment directories.
# No recursive whole-Drive scan is performed.

metadata_candidates = []


for candidate_name in [
    "metadata.csv",
    "eye_metadata.csv",
    "roi_metadata.csv",
]:

    candidate_paths = [
        EYE_ROOT / candidate_name,
        EYE_ROI_ROOT / candidate_name,
        EXPERIMENT_ROOT / candidate_name,
    ]

    for candidate_path in candidate_paths:

        if candidate_path.is_file():
            metadata_candidates.append(candidate_path)


# Remove duplicates while preserving order.
seen_metadata = set()
unique_metadata_candidates = []

for path in metadata_candidates:

    key = str(path)

    if key not in seen_metadata:
        seen_metadata.add(key)
        unique_metadata_candidates.append(path)


metadata_candidates = unique_metadata_candidates


# ============================================================
# 9. DEFINE METADATA PATH WHEN UNAMBIGUOUS
# ============================================================

ROI_METADATA_PATH = None


if len(metadata_candidates) == 1:

    ROI_METADATA_PATH = metadata_candidates[0]

elif len(metadata_candidates) > 1:

    # Do not silently guess between multiple metadata files.
    print(
        "\nWARNING: Multiple metadata files were found."
    )

    for path in metadata_candidates:
        print(" -", path)


# ============================================================
# 10. FINAL DIRECTORY REPORT
# ============================================================

print("\n" + "=" * 100)
print("GOOGLE DRIVE — EYE EXPERIMENT DIRECTORY VALIDATION")
print("=" * 100)

print(f"MyDrive          : {MY_DRIVE}")
print(f"AISC root        : {AISC_ROOT}")
print(f"Experiments root : {EXPERIMENTS_ROOT}")
print(f"User root        : {USER_ROOT}")
print(f"Experiment root  : {EXPERIMENT_ROOT}")
print(f"Eye root         : {EYE_ROOT}")
print(f"Eye ROI root     : {EYE_ROI_ROOT}")
print(f"Results root     : {RESULTS_ROOT}")

print("-" * 100)

print(
    f"Eye ROI root item count : {len(eye_roi_items)}"
)

print(
    f"Eye ROI directories     : {len(eye_roi_dirs)}"
)

print(
    f"Eye ROI files           : {len(eye_roi_files)}"
)


if eye_roi_dirs:

    print("\nEye ROI directories:")

    for path in eye_roi_dirs[:30]:
        print(f"  [DIR] {path.name}")

    if len(eye_roi_dirs) > 30:
        print(
            f"  ... +{len(eye_roi_dirs) - 30} more directories"
        )


if eye_roi_files:

    print("\nEye ROI root files:")

    for path in eye_roi_files[:30]:
        print(f"  [FILE] {path.name}")

    if len(eye_roi_files) > 30:
        print(
            f"  ... +{len(eye_roi_files) - 30} more files"
        )


print("-" * 100)

if ROI_METADATA_PATH is not None:

    print(
        "Metadata detected :",
        ROI_METADATA_PATH
    )

elif len(metadata_candidates) == 0:

    print(
        "Metadata detected : NOT FOUND AT ROOT LEVEL"
    )

    print(
        "Metadata detection will be handled "
        "explicitly in the next cell."
    )

else:

    print(
        "Metadata detected : MULTIPLE CANDIDATES"
    )

    print(
        "The next cell must select the correct metadata file explicitly."
    )


print("=" * 100)
print("CELL 2 COMPLETED SUCCESSFULLY")

Google Drive is already mounted.

GOOGLE DRIVE — EYE EXPERIMENT DIRECTORY VALIDATION
MyDrive          : /content/drive/MyDrive
AISC root        : /content/drive/MyDrive/AISC DeepFake Çalışmaları
Experiments root : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler
User root        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader
Experiment root  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
Eye root         : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz
Eye ROI root     : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
Results root     : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar
----------------------------------------------------------------------------------------------------
Eye ROI root item count : 5
Eye ROI directories     : 3
Eye ROI files           : 2

Eye ROI directories:
  [DIR] fake
  [DIR] real
  [DIR] Sonuçla

In [5]:
from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

print("MyDrive içeriği:")
for item in MY_DRIVE.iterdir():
    print(
        "NAME :", item.name,
        "\nPATH :", item,
        "\nDIR  :", item.is_dir(),
        "\n"
    )

MyDrive içeriği:
NAME : AISC DeepFake Çalışmaları 
PATH : /content/drive/MyDrive/AISC DeepFake Çalışmaları 
DIR  : True 



In [8]:
# ============================================================
# CELL 3 — EYE METADATA AUTO-DISCOVERY AND SCHEMA GATE
# ============================================================

from pathlib import Path
import pandas as pd

REQUIRED_SOURCE_COLUMNS = {
    "label",
    "split",
    "status",
    "combined_eye_path",
}

PREFERRED_ID_COLUMNS = {
    "sample_id",
    "video_id",
    "source_frame",
    "frame_stem",
    "face_id",
}

metadata_candidates = sorted(
    EYE_ROOT.rglob("*.csv"),
    key=lambda p: normalize_name(str(p))
)

valid_metadata_candidates = []

for candidate in metadata_candidates:
    try:
        header = pd.read_csv(candidate, nrows=0)
    except Exception:
        continue

    columns = set(header.columns)
    if REQUIRED_SOURCE_COLUMNS.issubset(columns):
        valid_metadata_candidates.append(candidate)

if not valid_metadata_candidates:
    raise FileNotFoundError(
        "No CSV containing the required eye metadata columns was found.\n"
        f"Required: {sorted(REQUIRED_SOURCE_COLUMNS)}\n"
        f"Search root: {EYE_ROOT}"
    )

def metadata_priority(path):
    name = path.name.casefold()
    score = 0
    if name == "metadata.csv":
        score -= 100
    if "backup" in name:
        score += 50
    if "manifest" in name:
        score += 10
    return (score, len(path.parts), normalize_name(str(path)))

MASTER_METADATA_PATH = sorted(
    valid_metadata_candidates,
    key=metadata_priority
)[0]

DATASET_ROOT = MASTER_METADATA_PATH.parent
master_metadata = pd.read_csv(MASTER_METADATA_PATH)

missing_columns = REQUIRED_SOURCE_COLUMNS.difference(master_metadata.columns)
if missing_columns:
    raise ValueError(
        f"Schema gate failed. Missing columns: {sorted(missing_columns)}"
    )

for col in ["label", "split", "status"]:
    master_metadata[col] = (
        master_metadata[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

allowed_labels = {"real", "fake"}
allowed_splits = {"train", "val", "test"}

unexpected_labels = set(master_metadata["label"].dropna()) - allowed_labels
unexpected_splits = set(master_metadata["split"].dropna()) - allowed_splits

if unexpected_labels:
    raise ValueError(f"Unexpected labels: {sorted(unexpected_labels)}")
if unexpected_splits:
    raise ValueError(f"Unexpected splits: {sorted(unexpected_splits)}")

print("\n" + "=" * 90)
print("EYE METADATA AUTO-DISCOVERY")
print("=" * 90)
print("Master metadata :", MASTER_METADATA_PATH)
print("Dataset root    :", DATASET_ROOT)
print("Shape           :", master_metadata.shape)
print("Columns         :", master_metadata.columns.tolist())
print("\nStatus counts:")
print(master_metadata["status"].value_counts(dropna=False).to_string())
print("\nSplit / label counts:")
print(pd.crosstab(master_metadata["split"], master_metadata["label"]).to_string())
print("=" * 90)
print("CELL 3 COMPLETED")



EYE METADATA AUTO-DISCOVERY
Master metadata : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
Dataset root    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
Shape           : (3097, 41)
Columns         : ['sample_id', 'source_frame', 'relative_frame_path', 'label', 'split', 'video_id', 'frame_stem', 'face_id', 'face_id_scope', 'faces_in_frame', 'left_eye_detected', 'right_eye_detected', 'left_eye_path', 'right_eye_path', 'combined_eye_path', 'debug_path', 'landmarks_path', 'left_bbox_x1', 'left_bbox_y1', 'left_bbox_x2', 'left_bbox_y2', 'right_bbox_x1', 'right_bbox_y1', 'right_bbox_x2', 'right_bbox_y2', 'combined_bbox_x1', 'combined_bbox_y1', 'combined_bbox_x2', 'combined_bbox_y2', 'left_eye_width_px', 'left_eye_height_px', 'right_eye_width_px', 'right_eye_height_px', 'left_iris_landmarks_available', 'right_iris_landmarks_available', 'left_iris_visible_estimate', 'right_iris_visible_es

In [10]:
# ============================================================
# CELL 4 — CANONICAL EYE MANIFEST, PATH RESOLUTION AND RUN DIRS
# ROBUST STATUS NORMALIZATION + EYE DATASET BUILD
# ============================================================

import hashlib
import json
import os
from pathlib import Path
from datetime import datetime

import pandas as pd


# ============================================================
# 1. CONSTANTS
# ============================================================

CLASS_NAMES = ["real", "fake"]

LABEL_MAP = {
    "real": 0,
    "fake": 1,
}

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ============================================================
# 2. REQUIRED METADATA COLUMNS
# ============================================================

required_columns = {
    "status",
    "split",
    "label",
    "combined_eye_path",
}

missing_columns = required_columns.difference(
    master_metadata.columns
)

if missing_columns:
    raise ValueError(
        "master_metadata is missing required columns: "
        f"{sorted(missing_columns)}"
    )


# ============================================================
# 3. NORMALIZE METADATA TEXT FIELDS
# ============================================================

master_metadata = master_metadata.copy()

master_metadata["status"] = (
    master_metadata["status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

master_metadata["split"] = (
    master_metadata["split"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

master_metadata["label"] = (
    master_metadata["label"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


# ============================================================
# 4. STATUS NORMALIZATION
# ============================================================

# Raw pipeline statuses are mapped into three canonical groups:
#
# SUCCESS -> usable ROI
# SKIPPED -> intentionally unusable / no valid ROI
# ERROR   -> actual processing failure

STATUS_MAP = {
    # -----------------------
    # SUCCESS-LIKE
    # -----------------------
    "SUCCESS": "SUCCESS",
    "OK": "SUCCESS",
    "DONE": "SUCCESS",
    "VALID": "SUCCESS",

    # -----------------------
    # SKIPPED-LIKE
    # -----------------------
    "SKIPPED": "SKIPPED",
    "NO_FACE": "SKIPPED",
    "NO_EYE": "SKIPPED",
    "NO_EYES": "SKIPPED",
    "NO_LANDMARK": "SKIPPED",
    "NO_LANDMARKS": "SKIPPED",
    "INVALID_ROI": "SKIPPED",
    "LOW_QUALITY": "SKIPPED",
    "BLUR": "SKIPPED",

    # -----------------------
    # ERROR-LIKE
    # -----------------------
    "ERROR": "ERROR",
    "FAILED": "ERROR",
    "FAIL": "ERROR",
    "EXCEPTION": "ERROR",
}


raw_status = master_metadata["status"]

canonical_status = raw_status.map(STATUS_MAP)


# ============================================================
# 5. UNKNOWN STATUS SAFETY GATE
# ============================================================

unknown_mask = canonical_status.isna()

if unknown_mask.any():

    unknown_values = sorted(
        raw_status.loc[unknown_mask]
        .dropna()
        .unique()
        .tolist()
    )

    unknown_counts = (
        raw_status.loc[unknown_mask]
        .value_counts()
        .to_dict()
    )

    raise ValueError(
        "\nUnknown status values were found in metadata.\n"
        f"Values : {unknown_values}\n"
        f"Counts : {unknown_counts}\n\n"
        "Add these statuses explicitly to STATUS_MAP before "
        "continuing. They will not be guessed automatically."
    )


master_metadata["canonical_status"] = canonical_status


# ============================================================
# 6. METADATA ACCOUNTING
# ============================================================

success_count = int(
    (canonical_status == "SUCCESS").sum()
)

skipped_count = int(
    (canonical_status == "SKIPPED").sum()
)

error_count = int(
    (canonical_status == "ERROR").sum()
)

total_metadata_records = int(
    len(master_metadata)
)


if (
    total_metadata_records
    != success_count + skipped_count + error_count
):
    raise AssertionError(
        "Metadata accounting equality failed:\n"
        f"{total_metadata_records} != "
        f"{success_count} + "
        f"{skipped_count} + "
        f"{error_count}"
    )


# ============================================================
# 7. DISPLAY RAW STATUS ACCOUNTING
# ============================================================

print("\nRaw status distribution:")

raw_status_counts = (
    raw_status
    .value_counts(dropna=False)
    .sort_index()
)

for status_name, count in raw_status_counts.items():
    print(
        f"  {str(status_name):20s}: {int(count):7d}"
    )


print("\nCanonical status distribution:")

print(
    f"  SUCCESS : {success_count}"
)

print(
    f"  SKIPPED : {skipped_count}"
)

print(
    f"  ERROR   : {error_count}"
)


# ============================================================
# 8. KEEP ONLY SUCCESSFUL EYE ROI RECORDS
# ============================================================

successful = (
    master_metadata.loc[
        master_metadata["canonical_status"] == "SUCCESS"
    ]
    .copy()
    .reset_index(drop=True)
)


if successful.empty:
    raise ValueError(
        "No usable SUCCESS / OK eye ROI rows were found."
    )


# ============================================================
# 9. VALIDATE LABELS
# ============================================================

allowed_labels = set(LABEL_MAP)

invalid_label_mask = ~successful["label"].isin(
    allowed_labels
)

if invalid_label_mask.any():

    invalid_labels = sorted(
        successful.loc[
            invalid_label_mask,
            "label"
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Invalid labels detected: {invalid_labels}"
    )


# ============================================================
# 10. VALIDATE SPLITS
# ============================================================

allowed_splits = {
    "train",
    "val",
    "test",
}

invalid_split_mask = ~successful["split"].isin(
    allowed_splits
)

if invalid_split_mask.any():

    invalid_splits = sorted(
        successful.loc[
            invalid_split_mask,
            "split"
        ]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Invalid split values detected: {invalid_splits}"
    )


# ============================================================
# 11. RESOLVE COMBINED EYE PATH
# ============================================================

def resolve_eye_path(raw_value):

    if pd.isna(raw_value):
        return None

    raw_text = str(raw_value).strip()

    if not raw_text:
        return None

    raw = Path(raw_text)


    direct_candidates = []


    # --------------------------------------------------------
    # Absolute path
    # --------------------------------------------------------

    if raw.is_absolute():

        direct_candidates.append(raw)


        # Old Colab / Drive path may exist in metadata.
        # Try to rebuild from recognizable project fragments.

        raw_parts = list(raw.parts)

        try:
            eye_index = raw_parts.index("Göz")

            relative_from_eye = Path(
                *raw_parts[eye_index + 1:]
            )

            direct_candidates.append(
                EYE_ROOT / relative_from_eye
            )

        except ValueError:
            pass


        try:
            roi_index = raw_parts.index(
                "eye_roi_output"
            )

            relative_from_roi = Path(
                *raw_parts[roi_index + 1:]
            )

            direct_candidates.append(
                EYE_ROI_ROOT
                / relative_from_roi
            )

        except ValueError:
            pass


    # --------------------------------------------------------
    # Relative path
    # --------------------------------------------------------

    else:

        direct_candidates.extend([
            EYE_ROI_ROOT / raw,
            EYE_ROOT / raw,
            EXPERIMENT_ROOT / raw,
            MY_DRIVE / raw,
        ])


    # --------------------------------------------------------
    # Test candidates
    # --------------------------------------------------------

    seen = set()

    for candidate in direct_candidates:

        candidate = Path(candidate)

        candidate_key = str(candidate)

        if candidate_key in seen:
            continue

        seen.add(candidate_key)

        if (
            candidate.exists()
            and candidate.is_file()
            and candidate.suffix.lower()
            in IMAGE_EXTENSIONS
        ):
            return candidate


    return None


successful["resolved_eye_path"] = (
    successful["combined_eye_path"]
    .map(resolve_eye_path)
)


# ============================================================
# 12. FALLBACK FILE RECOVERY
# ============================================================

missing_mask = (
    successful["resolved_eye_path"].isna()
)

missing_count = int(
    missing_mask.sum()
)


if missing_count:

    print(
        f"\nDirect path resolution failed for "
        f"{missing_count} SUCCESS records."
    )

    print(
        "Building fallback eye-image filename index..."
    )


    filename_index = {}

    ambiguous_names = set()


    for image_path in EYE_ROI_ROOT.rglob("*"):

        if not image_path.is_file():
            continue

        if (
            image_path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue


        filename = image_path.name


        if filename in filename_index:

            ambiguous_names.add(filename)

        else:

            filename_index[filename] = (
                image_path
            )


    # Remove ambiguous filenames completely.
    for filename in ambiguous_names:
        filename_index.pop(
            filename,
            None,
        )


    def recover_by_filename(raw_value):

        if pd.isna(raw_value):
            return None

        filename = Path(
            str(raw_value)
        ).name

        return filename_index.get(
            filename
        )


    successful.loc[
        missing_mask,
        "resolved_eye_path"
    ] = (
        successful.loc[
            missing_mask,
            "combined_eye_path"
        ]
        .map(recover_by_filename)
    )


# ============================================================
# 13. FINAL PATH QUALITY GATE
# ============================================================

still_missing = (
    successful["resolved_eye_path"].isna()
)


if still_missing.any():

    missing_examples = (
        successful.loc[
            still_missing,
            [
                "combined_eye_path",
                "label",
                "split",
            ]
        ]
        .head(20)
    )

    raise FileNotFoundError(
        "\nSome SUCCESS / OK combined-eye images "
        "could not be found on disk.\n"
        f"Missing count: "
        f"{int(still_missing.sum())}\n\n"
        "Examples:\n"
        f"{missing_examples.to_string(index=False)}"
    )


# ============================================================
# 14. STABLE CANONICAL SAMPLE ID
# ============================================================

def safe_str_series(
    column,
    default="",
):

    if column in successful.columns:

        return (
            successful[column]
            .fillna(default)
            .astype(str)
            .str.strip()
        )

    return pd.Series(
        [default] * len(successful),
        index=successful.index,
        dtype=str,
    )


original_sample_id = (
    safe_str_series("sample_id")
)

video_id = (
    safe_str_series("video_id")
)

source_frame = (
    safe_str_series("source_frame")
)

frame_stem = (
    safe_str_series("frame_stem")
)

face_id = (
    safe_str_series(
        "face_id",
        default="0",
    )
)

resolved_name = (
    successful["resolved_eye_path"]
    .map(
        lambda path:
        Path(path).name
    )
)


identity_key = (
    original_sample_id
    + "|"
    + video_id
    + "|"
    + source_frame
    + "|"
    + frame_stem
    + "|"
    + face_id
    + "|"
    + resolved_name
)


def stable_id(text):

    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()[:20]


canonical_sample_id = (
    original_sample_id.copy()
)


invalid_id = (
    canonical_sample_id.eq("")
    |
    canonical_sample_id.duplicated(
        keep=False
    )
)


canonical_sample_id.loc[
    invalid_id
] = (
    identity_key.loc[
        invalid_id
    ]
    .map(stable_id)
)


# ============================================================
# 15. DUPLICATE CANONICAL ID GATE
# ============================================================

if canonical_sample_id.duplicated().any():

    duplicate_mask = (
        canonical_sample_id.duplicated(
            keep=False
        )
    )

    duplicate_preview = (
        successful.loc[
            duplicate_mask
        ]
        .head(20)
    )

    raise ValueError(
        "\nCanonical sample_id is still duplicated "
        "after deterministic repair.\n"
        "This means multiple metadata rows describe "
        "the exact same identity.\n\n"
        f"{duplicate_preview.to_string(index=False)}"
    )


successful[
    "canonical_sample_id"
] = canonical_sample_id

successful[
    "original_sample_id"
] = original_sample_id


# ============================================================
# 16. DUPLICATE IMAGE PATH GATE
# ============================================================

resolved_path_text = (
    successful["resolved_eye_path"]
    .map(str)
)

duplicate_path_mask = (
    resolved_path_text.duplicated(
        keep=False
    )
)


if duplicate_path_mask.any():

    duplicate_path_preview = (
        successful.loc[
            duplicate_path_mask,
            [
                "canonical_sample_id",
                "label",
                "split",
                "combined_eye_path",
                "resolved_eye_path",
            ]
        ]
        .head(20)
    )

    raise ValueError(
        "\nThe same eye ROI image is referenced by "
        "multiple SUCCESS metadata rows.\n\n"
        f"{duplicate_path_preview.to_string(index=False)}"
    )


# ============================================================
# 17. BUILD DATASET RECORDS
# ============================================================

dataset_records = {}

actual_counts = {}


for split_name in [
    "train",
    "val",
    "test",
]:

    split_df = (
        successful.loc[
            successful["split"]
            == split_name
        ]
        .copy()
        .reset_index(drop=True)
    )


    if split_df.empty:

        raise ValueError(
            f"No SUCCESS records found "
            f"for split: {split_name}"
        )


    paths = (
        split_df[
            "resolved_eye_path"
        ]
        .map(str)
        .tolist()
    )


    labels = (
        split_df["label"]
        .map(LABEL_MAP)
        .astype(int)
        .tolist()
    )


    sample_ids = (
        split_df[
            "canonical_sample_id"
        ]
        .tolist()
    )


    dataset_records[
        split_name
    ] = {
        "paths": paths,
        "labels": labels,
        "sample_ids": sample_ids,
    }


    actual_counts[
        split_name
    ] = {

        "real": int(
            (
                split_df["label"]
                == "real"
            ).sum()
        ),

        "fake": int(
            (
                split_df["label"]
                == "fake"
            ).sum()
        ),

        "total": int(
            len(split_df)
        ),
    }


# ============================================================
# 18. PATH LEAKAGE CHECK
# ============================================================

train_paths = set(
    dataset_records["train"]["paths"]
)

val_paths = set(
    dataset_records["val"]["paths"]
)

test_paths = set(
    dataset_records["test"]["paths"]
)


if not train_paths.isdisjoint(
    val_paths
):
    raise AssertionError(
        "Path leakage detected: train vs val."
    )


if not train_paths.isdisjoint(
    test_paths
):
    raise AssertionError(
        "Path leakage detected: train vs test."
    )


if not val_paths.isdisjoint(
    test_paths
):
    raise AssertionError(
        "Path leakage detected: val vs test."
    )


# ============================================================
# 19. VIDEO-LEVEL LEAKAGE CHECK
# ============================================================

if "video_id" in successful.columns:

    split_video_ids = {}


    for split_name in [
        "train",
        "val",
        "test",
    ]:

        split_video_ids[
            split_name
        ] = set(
            successful.loc[
                successful["split"]
                == split_name,
                "video_id",
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .loc[
                lambda s:
                s.ne("")
            ]
            .tolist()
        )


    if not split_video_ids[
        "train"
    ].isdisjoint(
        split_video_ids["val"]
    ):

        overlap = (
            split_video_ids["train"]
            &
            split_video_ids["val"]
        )

        raise AssertionError(
            "Video leakage: train vs val.\n"
            f"Examples: {sorted(overlap)[:20]}"
        )


    if not split_video_ids[
        "train"
    ].isdisjoint(
        split_video_ids["test"]
    ):

        overlap = (
            split_video_ids["train"]
            &
            split_video_ids["test"]
        )

        raise AssertionError(
            "Video leakage: train vs test.\n"
            f"Examples: {sorted(overlap)[:20]}"
        )


    if not split_video_ids[
        "val"
    ].isdisjoint(
        split_video_ids["test"]
    ):

        overlap = (
            split_video_ids["val"]
            &
            split_video_ids["test"]
        )

        raise AssertionError(
            "Video leakage: val vs test.\n"
            f"Examples: {sorted(overlap)[:20]}"
        )


# ============================================================
# 20. RUN ID
# ============================================================

resume_run_id = (
    CONFIG.get(
        "resume_run_id"
    )
)


if resume_run_id:

    RUN_ID = str(
        resume_run_id
    ).strip()

else:

    RUN_ID = (
        datetime.now()
        .strftime(
            "%Y%m%d_%H%M"
        )
        + f"_eye_densenet121_seed{SEED}"
    )


# ============================================================
# 21. RUN DIRECTORY STRUCTURE
# ============================================================

MODEL_RESULTS_ROOT = (
    RESULTS_ROOT
    / "DenseNet121_Eye_Results"
)

RUN_ROOT = (
    MODEL_RESULTS_ROOT
    / RUN_ID
)


CHECKPOINT_DIR = (
    RUN_ROOT
    / "checkpoints"
)

LOG_DIR = (
    RUN_ROOT
    / "logs"
)

METRICS_DIR = (
    RUN_ROOT
    / "metrics"
)

PREDICTION_DIR = (
    RUN_ROOT
    / "predictions"
)

FIGURE_DIR = (
    RUN_ROOT
    / "figures"
)

METADATA_DIR = (
    RUN_ROOT
    / "metadata"
)


for folder in [
    MODEL_RESULTS_ROOT,
    RUN_ROOT,
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    PREDICTION_DIR,
    FIGURE_DIR,
    METADATA_DIR,
]:

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


MODEL_DIR = CHECKPOINT_DIR
TABLE_DIR = METRICS_DIR


# ============================================================
# 22. SAVE CANONICAL SUCCESS MANIFEST
# ============================================================

canonical_manifest_path = (
    METADATA_DIR
    / "canonical_eye_manifest.csv"
)


manifest_to_save = (
    successful.copy()
)

manifest_to_save[
    "resolved_eye_path"
] = (
    manifest_to_save[
        "resolved_eye_path"
    ]
    .map(str)
)


manifest_to_save.to_csv(
    canonical_manifest_path,
    index=False,
)


# ============================================================
# 23. SAVE DATASET ACCOUNTING
# ============================================================

accounting_payload = {

    "raw_status_counts": {
        str(key): int(value)
        for key, value
        in raw_status_counts.items()
    },

    "canonical_status_counts": {
        "SUCCESS": success_count,
        "SKIPPED": skipped_count,
        "ERROR": error_count,
    },

    "dataset_counts": actual_counts,

    "metadata_records": (
        total_metadata_records
    ),

    "canonical_success_records": (
        int(len(successful))
    ),

    "run_id": RUN_ID,
}


accounting_path = (
    METADATA_DIR
    / "dataset_accounting.json"
)


with open(
    accounting_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        accounting_payload,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 24. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("CANONICAL EYE DATASET ACCOUNTING")
print("=" * 100)


for split_name in [
    "train",
    "val",
    "test",
]:

    counts = (
        actual_counts[
            split_name
        ]
    )

    print(
        f"{split_name.upper():5} | "
        f"Real: {counts['real']:6d} | "
        f"Fake: {counts['fake']:6d} | "
        f"Total: {counts['total']:6d}"
    )


print("-" * 100)

print(
    "Metadata accounting : "
    f"{total_metadata_records} = "
    f"{success_count} usable + "
    f"{skipped_count} skipped + "
    f"{error_count} error"
)

print(
    "Usable raw statuses :",
    sorted(
        master_metadata.loc[
            master_metadata[
                "canonical_status"
            ]
            == "SUCCESS",
            "status",
        ]
        .unique()
        .tolist()
    )
)

print(
    "Canonical samples   :",
    successful[
        "canonical_sample_id"
    ].nunique()
)

print(
    "Resolved eye images :",
    successful[
        "resolved_eye_path"
    ].nunique()
)

print(
    "Run ID              :",
    RUN_ID
)

print(
    "Run root            :",
    RUN_ROOT
)

print(
    "Canonical manifest  :",
    canonical_manifest_path
)

print(
    "Accounting JSON     :",
    accounting_path
)

print("=" * 100)
print("CELL 4 COMPLETED SUCCESSFULLY")


Raw status distribution:
  NO_FACE             :     111
  OK                  :    2986

Canonical status distribution:
  SUCCESS : 2986
  SKIPPED : 111
  ERROR   : 0

CANONICAL EYE DATASET ACCOUNTING
TRAIN | Real:   1197 | Fake:   1191 | Total:   2388
VAL   | Real:    155 | Fake:    141 | Total:    296
TEST  | Real:    146 | Fake:    156 | Total:    302
----------------------------------------------------------------------------------------------------
Metadata accounting : 3097 = 2986 usable + 111 skipped + 0 error
Usable raw statuses : ['OK']
Canonical samples   : 2986
Resolved eye images : 2986
Run ID              : 20260808_1214_eye_densenet121_seed42
Run root            : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42
Canonical manifest  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/metad

In [11]:
# ============================================================
# CELL 5 — TF.DATA PIPELINE AND DATA AUGMENTATION
# ============================================================

import tensorflow as tf
from tensorflow.keras.applications.densenet import preprocess_input

# ------------------------------------------------------------
# Image and batch configuration
# ------------------------------------------------------------
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_SIZE = (IMAGE_HEIGHT, IMAGE_WIDTH)

BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

print("\n" + "=" * 75)
print("TF.DATA PIPELINE AND DENSENET121 PREPROCESSING")
print("=" * 75)

print("Image size :", IMAGE_SIZE)
print("Batch size :", BATCH_SIZE)


# ------------------------------------------------------------
# Controlled data augmentation
# Only for the training dataset
# ------------------------------------------------------------
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip(
            mode="horizontal",
            seed=SEED
        ),

        tf.keras.layers.RandomRotation(
            factor=0.03,
            fill_mode="reflect",
            seed=SEED
        ),

        tf.keras.layers.RandomZoom(
            height_factor=(-0.08, 0.08),
            width_factor=(-0.08, 0.08),
            fill_mode="reflect",
            seed=SEED
        ),

        tf.keras.layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="reflect",
            seed=SEED
        ),

        tf.keras.layers.RandomContrast(
            factor=0.10,
            seed=SEED
        ),
    ],
    name="eye_data_augmentation"
)


# ------------------------------------------------------------
# Image decoding function
# ------------------------------------------------------------
def decode_and_resize_image(image_path, label):
    """
    Reads an image, converts it to RGB and resizes it to
    the DenseNet121 input resolution.
    """

    image_bytes = tf.io.read_file(image_path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        size=IMAGE_SIZE,
        method=tf.image.ResizeMethod.BILINEAR,
        antialias=True
    )

    image = tf.cast(
        image,
        tf.float32
    )

    label = tf.cast(
        label,
        tf.float32
    )

    return image, label


# ------------------------------------------------------------
# Training preprocessing
# ------------------------------------------------------------
def preprocess_training_image(image, label):
    """
    Applies augmentation and DenseNet121 preprocessing.
    """

    image = data_augmentation(
        image,
        training=True
    )

    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    image = preprocess_input(image)

    return image, label


# ------------------------------------------------------------
# Validation and test preprocessing
# ------------------------------------------------------------
def preprocess_evaluation_image(image, label):
    """
    Applies only DenseNet121 preprocessing.
    No augmentation is used.
    """

    image = preprocess_input(image)

    return image, label


# ------------------------------------------------------------
# Dataset creation function
# ------------------------------------------------------------
def create_tf_dataset(
    image_paths,
    labels,
    training=False
):
    """
    Creates a TensorFlow dataset from image paths and labels.
    """

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            image_paths,
            labels
        )
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(image_paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        decode_and_resize_image,
        num_parallel_calls=AUTOTUNE,
        deterministic=not training
    )

    if training:
        dataset = dataset.map(
            preprocess_training_image,
            num_parallel_calls=AUTOTUNE,
            deterministic=False
        )

    else:
        dataset = dataset.map(
            preprocess_evaluation_image,
            num_parallel_calls=AUTOTUNE,
            deterministic=True
        )

    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=False
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset


# ------------------------------------------------------------
# Create train, validation and test datasets
# ------------------------------------------------------------
train_dataset = create_tf_dataset(
    dataset_records["train"]["paths"],
    dataset_records["train"]["labels"],
    training=True
)

val_dataset = create_tf_dataset(
    dataset_records["val"]["paths"],
    dataset_records["val"]["labels"],
    training=False
)

test_dataset = create_tf_dataset(
    dataset_records["test"]["paths"],
    dataset_records["test"]["labels"],
    training=False
)


# ------------------------------------------------------------
# Dataset cardinalities
# ------------------------------------------------------------
train_batches = int(
    tf.data.experimental.cardinality(
        train_dataset
    ).numpy()
)

val_batches = int(
    tf.data.experimental.cardinality(
        val_dataset
    ).numpy()
)

test_batches = int(
    tf.data.experimental.cardinality(
        test_dataset
    ).numpy()
)


# ------------------------------------------------------------
# Read one validation batch for smoke testing
# ------------------------------------------------------------
sample_images, sample_labels = next(
    iter(val_dataset)
)

print("\nDataset batches:")
print("Train batches      :", train_batches)
print("Validation batches :", val_batches)
print("Test batches       :", test_batches)

print("\nSmoke-test batch:")
print("Image batch shape  :", sample_images.shape)
print("Label batch shape  :", sample_labels.shape)
print("Image dtype        :", sample_images.dtype)
print("Label dtype        :", sample_labels.dtype)

print(
    "Preprocessed range :",
    float(tf.reduce_min(sample_images)),
    "to",
    float(tf.reduce_max(sample_images))
)

print(
    "Sample labels      :",
    sample_labels.numpy().astype(int).tolist()
)


# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------
assert sample_images.shape[1:] == (
    IMAGE_HEIGHT,
    IMAGE_WIDTH,
    3
), "Image shape is incorrect."

assert sample_labels.shape[0] <= BATCH_SIZE, (
    "Batch size is incorrect."
)

expected_train_batches = int(np.ceil(actual_counts["train"]["total"] / BATCH_SIZE))
expected_val_batches = int(np.ceil(actual_counts["val"]["total"] / BATCH_SIZE))
expected_test_batches = int(np.ceil(actual_counts["test"]["total"] / BATCH_SIZE))

assert train_batches == expected_train_batches, (
    f"Unexpected number of train batches: {train_batches} "
    f"(expected {expected_train_batches})"
)
assert val_batches == expected_val_batches, (
    f"Unexpected number of validation batches: {val_batches} "
    f"(expected {expected_val_batches})"
)
assert test_batches == expected_test_batches, (
    f"Unexpected number of test batches: {test_batches} "
    f"(expected {expected_test_batches})"
)


print("-" * 75)
print("Training augmentation : ENABLED")
print("Validation augmentation: DISABLED")
print("Test augmentation      : DISABLED")
print("DenseNet preprocessing : ENABLED")
print("Dataset smoke test     : PASS")
print("=" * 75)
print("CELL 5 COMPLETED")



TF.DATA PIPELINE AND DENSENET121 PREPROCESSING
Image size : (224, 224)
Batch size : 16

Dataset batches:
Train batches      : 150
Validation batches : 19
Test batches       : 19

Smoke-test batch:
Image batch shape  : (16, 224, 224, 3)
Label batch shape  : (16,)
Image dtype        : <dtype: 'float32'>
Label dtype        : <dtype: 'float32'>
Preprocessed range : -2.1179039478302 to 2.248908281326294
Sample labels      : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
---------------------------------------------------------------------------
Training augmentation : ENABLED
Validation augmentation: DISABLED
Test augmentation      : DISABLED
DenseNet preprocessing : ENABLED
Dataset smoke test     : PASS
CELL 5 COMPLETED


In [12]:
# ============================================================
# CELL 6 — SOURCE / SAMPLE ID AUDIT
# ============================================================

from pathlib import Path

print("\n" + "=" * 88)
print("EYE SAMPLE ID AND SOURCE AUDIT")
print("=" * 88)

assert successful["canonical_sample_id"].notna().all()
assert successful["canonical_sample_id"].is_unique

all_model_paths = []
for split_name in ["train", "val", "test"]:
    all_model_paths.extend(dataset_records[split_name]["paths"])

assert len(all_model_paths) == success_count, (
    "SUCCESS metadata count and model image count do not match."
)
assert len(all_model_paths) == len(set(all_model_paths)), (
    "Duplicate image paths exist in canonical SUCCESS data."
)

print("SUCCESS model images :", len(all_model_paths))
print("Unique sample IDs    :", successful["canonical_sample_id"].nunique())
print("Unique image paths   :", len(set(all_model_paths)))

if "video_id" in successful.columns:
    print("Unique source videos :", successful["video_id"].astype(str).nunique())
else:
    print("Unique source videos : video_id column not available")

print("=" * 88)
print("CELL 6 COMPLETED")



EYE SAMPLE ID AND SOURCE AUDIT
SUCCESS model images : 2986
Unique sample IDs    : 2986
Unique image paths   : 2986
Unique source videos : 6
CELL 6 COMPLETED


In [13]:
# ============================================================
# CELL 7 — SOURCE VIDEO AND SPLIT LEAKAGE PRECHECK
# ============================================================

print("\n" + "=" * 88)
print("SOURCE VIDEO / SPLIT PRECHECK")
print("=" * 88)

SOURCE_COLUMN_CANDIDATES = [
    "video_id",
    "source_video",
    "video",
    "video_path",
]

source_video_column = next(
    (c for c in SOURCE_COLUMN_CANDIDATES if c in successful.columns),
    None
)

if source_video_column is None:
    raise ValueError(
        "A source-video identifier is required for leakage control. "
        f"Expected one of: {SOURCE_COLUMN_CANDIDATES}"
    )

successful["source_video"] = (
    successful[source_video_column]
    .astype(str)
    .str.strip()
)

if successful["source_video"].eq("").any():
    raise ValueError("Blank source_video values exist in SUCCESS metadata.")

source_split_counts = (
    successful.groupby("source_video")["split"]
    .nunique()
)

leaking_sources = source_split_counts[source_split_counts > 1]
if not leaking_sources.empty:
    raise AssertionError(
        "Source-video leakage detected. The same video appears in "
        "more than one split.\n"
        f"Examples:\n{leaking_sources.head(20).to_string()}"
    )

train_video_ids = set(
    successful.loc[successful["split"] == "train", "source_video"]
)
val_video_ids = set(
    successful.loc[successful["split"] == "val", "source_video"]
)
test_video_ids = set(
    successful.loc[successful["split"] == "test", "source_video"]
)

assert train_video_ids.isdisjoint(val_video_ids)
assert train_video_ids.isdisjoint(test_video_ids)
assert val_video_ids.isdisjoint(test_video_ids)

print("Train source videos :", len(train_video_ids))
print("Val source videos   :", len(val_video_ids))
print("Test source videos  :", len(test_video_ids))
print("Source leakage      : NONE")
print("=" * 88)
print("CELL 7 COMPLETED")



SOURCE VIDEO / SPLIT PRECHECK
Train source videos : 2
Val source videos   : 2
Test source videos  : 2
Source leakage      : NONE
CELL 7 COMPLETED


In [14]:
# ============================================================
# CELL 8 — MASTER METADATA AUDIT SUMMARY
# ============================================================

import json

print("\n" + "=" * 90)
print("MASTER EYE METADATA AUDIT SUMMARY")
print("=" * 90)
print("Metadata file:", MASTER_METADATA_PATH)
print("Rows         :", len(master_metadata))
print("SUCCESS      :", success_count)
print("SKIPPED      :", skipped_count)
print("ERROR        :", error_count)
print("\nCanonical split / label table:")
print(
    pd.crosstab(
        successful["split"],
        successful["label"]
    ).to_string()
)

optional_eye_columns = [
    "left_eye_detected",
    "right_eye_detected",
    "left_eye_path",
    "right_eye_path",
    "landmarks_path",
]
present_optional = [c for c in optional_eye_columns if c in successful.columns]
print("\nEye-specific metadata columns present:")
print(present_optional)

print("=" * 90)
print("CELL 8 COMPLETED")



MASTER EYE METADATA AUDIT SUMMARY
Metadata file: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
Rows         : 3097
SUCCESS      : 2986
SKIPPED      : 111
ERROR        : 0

Canonical split / label table:
label  fake  real
split            
test    156   146
train  1191  1197
val     141   155

Eye-specific metadata columns present:
['left_eye_detected', 'right_eye_detected', 'left_eye_path', 'right_eye_path', 'landmarks_path']
CELL 8 COMPLETED


In [16]:
# ============================================================
# CELL 9 — STANDARD MANIFEST, HASH AND LEAKAGE QUALITY GATES
# RESUMABLE SHA-256 CACHE VERSION
# ============================================================

import hashlib
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd


print("\n" + "=" * 100)
print("STANDARD EYE MANIFEST, HASH AND LEAKAGE QUALITY GATES")
print("=" * 100)


# ============================================================
# 1. ATOMIC WRITE HELPERS
# ============================================================

def atomic_write_json(output_path, payload):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    tmp = output_path.with_name(
        output_path.name + ".tmp"
    )

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=4,
            ensure_ascii=False,
        )

    os.replace(
        tmp,
        output_path,
    )


def atomic_write_csv(dataframe, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    tmp = output_path.with_name(
        output_path.name + ".tmp"
    )

    dataframe.to_csv(
        tmp,
        index=False,
    )

    os.replace(
        tmp,
        output_path,
    )


# ============================================================
# 2. SHA-256 HELPER
# ============================================================

def calculate_sha256(
    file_path,
    chunk_size=4 * 1024 * 1024,
):
    """
    Streaming SHA-256 calculation.

    Uses 4 MB chunks to reduce Drive read overhead.
    """

    file_path = Path(file_path)

    if not file_path.is_file():
        raise FileNotFoundError(
            f"Cannot hash missing file: {file_path}"
        )

    hasher = hashlib.sha256()

    with open(
        file_path,
        "rb",
        buffering=chunk_size,
    ) as stream:

        while True:
            chunk = stream.read(chunk_size)

            if not chunk:
                break

            hasher.update(chunk)

    return hasher.hexdigest()


# ============================================================
# 3. FRAME INDEX EXTRACTION
# ============================================================

def extract_frame_index(row):

    # Prefer explicit numeric fields.
    for column in [
        "frame_index",
        "frame_id",
        "frame_number",
    ]:

        if (
            column in row.index
            and pd.notna(row[column])
        ):

            try:
                return int(row[column])

            except (
                TypeError,
                ValueError,
            ):
                pass


    # Fall back to filename-like fields.
    for column in [
        "source_frame",
        "frame_stem",
    ]:

        if (
            column in row.index
            and pd.notna(row[column])
        ):

            match = re.search(
                r"(\d+)(?!.*\d)",
                str(row[column]),
            )

            if match:
                return int(
                    match.group(1)
                )


    raise ValueError(
        "Frame index could not be derived "
        "from metadata row:\n"
        f"{row.to_dict()}"
    )


# ============================================================
# 4. HASH CACHE PATH
# ============================================================

HASH_CACHE_PATH = (
    METADATA_DIR
    / "sha256_cache.csv"
)


# ============================================================
# 5. BUILD CURRENT FILE TABLE
# ============================================================

current_files = pd.DataFrame({
    "output_path": (
        successful[
            "resolved_eye_path"
        ]
        .map(str)
    )
})


if current_files[
    "output_path"
].duplicated().any():

    duplicate_examples = (
        current_files.loc[
            current_files[
                "output_path"
            ].duplicated(
                keep=False
            ),
            "output_path",
        ]
        .head(20)
        .tolist()
    )

    raise ValueError(
        "Duplicate resolved eye paths "
        "were found before hashing.\n"
        f"Examples: {duplicate_examples}"
    )


# ============================================================
# 6. LOAD EXISTING HASH CACHE
# ============================================================

hash_cache = {}


if HASH_CACHE_PATH.is_file():

    print(
        "\nExisting SHA-256 cache found:"
    )

    print(
        HASH_CACHE_PATH
    )

    existing_cache = pd.read_csv(
        HASH_CACHE_PATH
    )


    required_cache_columns = {
        "output_path",
        "file_size",
        "mtime_ns",
        "sha256",
    }

    missing_cache_columns = (
        required_cache_columns.difference(
            existing_cache.columns
        )
    )


    if missing_cache_columns:

        print(
            "Existing hash cache schema is old "
            "or invalid. It will be rebuilt."
        )

        existing_cache = pd.DataFrame()


    if not existing_cache.empty:

        for row in (
            existing_cache
            .itertuples(
                index=False
            )
        ):

            hash_cache[
                str(row.output_path)
            ] = {
                "file_size": int(
                    row.file_size
                ),
                "mtime_ns": int(
                    row.mtime_ns
                ),
                "sha256": str(
                    row.sha256
                ),
            }


print(
    f"Cached hash entries: "
    f"{len(hash_cache):,}"
)


# ============================================================
# 7. RESUMABLE SHA-256 PROCESS
# ============================================================

sha256_values = []

new_hashes = 0
reused_hashes = 0

CACHE_SAVE_INTERVAL = 100


print(
    f"\nPreparing SHA-256 for "
    f"{len(current_files):,} combined-eye images..."
)


for index, output_path in enumerate(
    current_files["output_path"],
    start=1,
):

    path = Path(output_path)


    if not path.is_file():

        raise FileNotFoundError(
            "SUCCESS image missing on disk:\n"
            f"{path}"
        )


    stat = path.stat()

    file_size = int(
        stat.st_size
    )

    mtime_ns = int(
        stat.st_mtime_ns
    )


    cached = hash_cache.get(
        output_path
    )


    cache_valid = (
        cached is not None
        and cached[
            "file_size"
        ] == file_size
        and cached[
            "mtime_ns"
        ] == mtime_ns
        and len(
            cached[
                "sha256"
            ]
        ) == 64
    )


    if cache_valid:

        sha_value = cached[
            "sha256"
        ]

        reused_hashes += 1


    else:

        sha_value = (
            calculate_sha256(
                path
            )
        )

        hash_cache[
            output_path
        ] = {
            "file_size": file_size,
            "mtime_ns": mtime_ns,
            "sha256": sha_value,
        }

        new_hashes += 1


    sha256_values.append(
        sha_value
    )


    # --------------------------------------------------------
    # Periodic atomic cache save
    # --------------------------------------------------------

    if (
        index % CACHE_SAVE_INTERVAL == 0
        or index == len(current_files)
    ):

        cache_dataframe = pd.DataFrame([
            {
                "output_path": path_key,
                "file_size": value[
                    "file_size"
                ],
                "mtime_ns": value[
                    "mtime_ns"
                ],
                "sha256": value[
                    "sha256"
                ],
            }
            for path_key, value
            in hash_cache.items()
        ])


        atomic_write_csv(
            cache_dataframe,
            HASH_CACHE_PATH,
        )


        print(
            f"Hashed/checkpointed: "
            f"{index:,} / "
            f"{len(current_files):,} | "
            f"new={new_hashes:,} | "
            f"reused={reused_hashes:,}"
        )


# ============================================================
# 8. SHA-256 QUALITY GATE
# ============================================================

sha256_series = pd.Series(
    sha256_values,
    dtype=str,
)


if not sha256_series.str.len().eq(
    64
).all():

    raise ValueError(
        "One or more SHA-256 values "
        "are invalid."
    )


# ============================================================
# 9. FRAME INDICES
# ============================================================

print(
    "\nExtracting frame indices..."
)


frame_indices = (
    successful
    .apply(
        extract_frame_index,
        axis=1,
    )
    .astype(int)
)


# ============================================================
# 10. FACE INDICES
# ============================================================

if "face_id" in successful.columns:

    face_indices = (
        pd.to_numeric(
            successful["face_id"],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

else:

    face_indices = pd.Series(
        np.zeros(
            len(successful),
            dtype=int,
        ),
        index=successful.index,
    )


# ============================================================
# 11. SOURCE VIDEO RESOLUTION
# ============================================================

if "source_video" in successful.columns:

    source_video_series = (
        successful[
            "source_video"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

elif "video_id" in successful.columns:

    source_video_series = (
        successful[
            "video_id"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

else:

    raise ValueError(
        "Neither source_video nor video_id "
        "exists in successful metadata."
    )


if source_video_series.eq("").any():

    empty_count = int(
        source_video_series
        .eq("")
        .sum()
    )

    raise ValueError(
        f"{empty_count} SUCCESS rows have "
        "an empty source-video identifier."
    )


# ============================================================
# 12. ROI STATE
# ============================================================

if "eye_state" in successful.columns:

    roi_state_series = (
        successful[
            "eye_state"
        ]
        .fillna(
            "not_recorded"
        )
        .astype(str)
    )

else:

    roi_state_series = pd.Series(
        [
            "not_recorded"
        ] * len(successful),
        index=successful.index,
        dtype=str,
    )


# ============================================================
# 13. BUILD STANDARD MODEL MANIFEST
# ============================================================

model_manifest = pd.DataFrame({

    "sample_id": (
        successful[
            "canonical_sample_id"
        ]
        .astype(str)
    ),

    "original_sample_id": (
        successful[
            "original_sample_id"
        ]
        .astype(str)
    ),

    "source_video": (
        source_video_series
    ),

    "frame_index": (
        frame_indices
    ),

    "face_index": (
        face_indices
    ),

    "roi_state": (
        roi_state_series
    ),

    "label": (
        successful[
            "label"
        ]
        .astype(str)
    ),

    "split": (
        successful[
            "split"
        ]
        .astype(str)
    ),

    "status": (
        "SUCCESS"
    ),

    "skip_reason": (
        ""
    ),

    "sha256": (
        sha256_values
    ),

    "output_path": (
        successful[
            "resolved_eye_path"
        ]
        .map(str)
    ),

    "run_id": (
        RUN_ID
    ),
})


# ============================================================
# 14. ADD OPTIONAL ORIGINAL METADATA COLUMNS
# ============================================================

for optional_col in [

    "source_frame",
    "frame_stem",

    "left_eye_detected",
    "right_eye_detected",

    "left_eye_path",
    "right_eye_path",

    "combined_eye_path",

    "landmarks_path",

]:

    if optional_col in successful.columns:

        model_manifest[
            optional_col
        ] = (
            successful[
                optional_col
            ]
            .values
        )


# ============================================================
# 15. REQUIRED MANIFEST SCHEMA
# ============================================================

REQUIRED_MANIFEST_COLUMNS = {

    "sample_id",
    "source_video",

    "frame_index",
    "face_index",

    "roi_state",

    "label",
    "split",

    "status",
    "skip_reason",

    "sha256",
    "output_path",

    "run_id",
}


missing_columns = (
    REQUIRED_MANIFEST_COLUMNS.difference(
        model_manifest.columns
    )
)


if missing_columns:

    raise ValueError(
        "Missing manifest columns: "
        f"{sorted(missing_columns)}"
    )


# ============================================================
# 16. MANIFEST QUALITY GATES
# ============================================================

if not model_manifest[
    "sample_id"
].is_unique:

    raise ValueError(
        "Duplicate canonical sample_id detected."
    )


if model_manifest[
    "output_path"
].isna().any():

    raise ValueError(
        "Missing output_path detected."
    )


if not model_manifest[
    "sha256"
].str.len().eq(
    64
).all():

    raise ValueError(
        "Invalid SHA-256 detected."
    )


missing_disk_files = [
    path
    for path
    in model_manifest[
        "output_path"
    ]
    if not Path(path).is_file()
]


if missing_disk_files:

    raise FileNotFoundError(
        "SUCCESS images are missing "
        "from disk.\n"
        f"Count: {len(missing_disk_files)}\n"
        f"Examples: "
        f"{missing_disk_files[:20]}"
    )


# ============================================================
# 17. SOURCE-VIDEO SPLIT ISOLATION
# ============================================================

split_sources = {

    split: set(
        model_manifest.loc[
            model_manifest[
                "split"
            ] == split,
            "source_video",
        ]
    )

    for split in [
        "train",
        "val",
        "test",
    ]
}


video_overlap_counts = {

    "train_vs_val": len(
        split_sources[
            "train"
        ]
        & split_sources[
            "val"
        ]
    ),

    "train_vs_test": len(
        split_sources[
            "train"
        ]
        & split_sources[
            "test"
        ]
    ),

    "val_vs_test": len(
        split_sources[
            "val"
        ]
        & split_sources[
            "test"
        ]
    ),
}


if any(
    video_overlap_counts.values()
):

    raise AssertionError(
        "Source-video leakage detected: "
        f"{video_overlap_counts}"
    )


# ============================================================
# 18. CONTENT-HASH SPLIT ISOLATION
# ============================================================

split_hashes = {

    split: set(
        model_manifest.loc[
            model_manifest[
                "split"
            ] == split,
            "sha256",
        ]
    )

    for split in [
        "train",
        "val",
        "test",
    ]
}


hash_overlap_counts = {

    "train_vs_val": len(
        split_hashes[
            "train"
        ]
        & split_hashes[
            "val"
        ]
    ),

    "train_vs_test": len(
        split_hashes[
            "train"
        ]
        & split_hashes[
            "test"
        ]
    ),

    "val_vs_test": len(
        split_hashes[
            "val"
        ]
        & split_hashes[
            "test"
        ]
    ),
}


if any(
    hash_overlap_counts.values()
):

    raise AssertionError(
        "Image-content leakage detected: "
        f"{hash_overlap_counts}"
    )


# ============================================================
# 19. OUTPUT PATHS
# ============================================================

MANIFEST_PATH = (
    METADATA_DIR
    / "model_manifest.csv"
)

LEAKAGE_REPORT_PATH = (
    METADATA_DIR
    / "leakage_check.csv"
)

ACCOUNTING_REPORT_PATH = (
    METADATA_DIR
    / "accounting_summary.json"
)


# ============================================================
# 20. LEAKAGE REPORT
# ============================================================

leakage_report = pd.DataFrame([

    {

        "comparison": key,

        "source_video_overlap": (
            video_overlap_counts[
                key
            ]
        ),

        "sha256_overlap": (
            hash_overlap_counts[
                key
            ]
        ),

    }

    for key in [
        "train_vs_val",
        "train_vs_test",
        "val_vs_test",
    ]
])


# ============================================================
# 21. ACCOUNTING SUMMARY
# ============================================================

accounting_summary = {

    "run_id": RUN_ID,

    "total_metadata_records": (
        total_metadata_records
    ),

    "success_count": (
        success_count
    ),

    "skipped_count": (
        skipped_count
    ),

    "error_count": (
        error_count
    ),

    "model_image_count": int(
        len(model_manifest)
    ),

    "canonical_sample_id_unique": bool(
        model_manifest[
            "sample_id"
        ].is_unique
    ),

    "source_video_overlap_counts": (
        video_overlap_counts
    ),

    "sha256_overlap_counts": (
        hash_overlap_counts
    ),

    "sha256_cache_file": str(
        HASH_CACHE_PATH
    ),

    "sha256_reused": int(
        reused_hashes
    ),

    "sha256_calculated": int(
        new_hashes
    ),
}


# ============================================================
# 22. ATOMIC SAVE REPORTS
# ============================================================

atomic_write_csv(
    model_manifest,
    MANIFEST_PATH,
)

atomic_write_csv(
    leakage_report,
    LEAKAGE_REPORT_PATH,
)

atomic_write_json(
    ACCOUNTING_REPORT_PATH,
    accounting_summary,
)


# ============================================================
# 23. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)

print(
    "MANIFEST / HASH / LEAKAGE QUALITY GATES PASSED"
)

print("=" * 100)


print(
    "Manifest rows       :",
    len(model_manifest)
)

print(
    "Unique sample IDs   :",
    model_manifest[
        "sample_id"
    ].nunique()
)

print(
    "SHA-256 calculated  :",
    new_hashes
)

print(
    "SHA-256 reused      :",
    reused_hashes
)

print(
    "Source overlaps     :",
    video_overlap_counts
)

print(
    "Hash overlaps       :",
    hash_overlap_counts
)

print(
    "Hash cache          :",
    HASH_CACHE_PATH
)

print(
    "Manifest            :",
    MANIFEST_PATH
)

print(
    "Leakage report      :",
    LEAKAGE_REPORT_PATH
)

print(
    "Accounting report   :",
    ACCOUNTING_REPORT_PATH
)

print("=" * 100)
print("CELL 9 COMPLETED SUCCESSFULLY")


STANDARD EYE MANIFEST, HASH AND LEAKAGE QUALITY GATES
Cached hash entries: 0

Preparing SHA-256 for 2,986 combined-eye images...
Hashed/checkpointed: 100 / 2,986 | new=100 | reused=0
Hashed/checkpointed: 200 / 2,986 | new=200 | reused=0
Hashed/checkpointed: 300 / 2,986 | new=300 | reused=0
Hashed/checkpointed: 400 / 2,986 | new=400 | reused=0
Hashed/checkpointed: 500 / 2,986 | new=500 | reused=0
Hashed/checkpointed: 600 / 2,986 | new=600 | reused=0
Hashed/checkpointed: 700 / 2,986 | new=700 | reused=0
Hashed/checkpointed: 800 / 2,986 | new=800 | reused=0
Hashed/checkpointed: 900 / 2,986 | new=900 | reused=0
Hashed/checkpointed: 1,000 / 2,986 | new=1,000 | reused=0
Hashed/checkpointed: 1,100 / 2,986 | new=1,100 | reused=0
Hashed/checkpointed: 1,200 / 2,986 | new=1,200 | reused=0
Hashed/checkpointed: 1,300 / 2,986 | new=1,300 | reused=0
Hashed/checkpointed: 1,400 / 2,986 | new=1,400 | reused=0
Hashed/checkpointed: 1,500 / 2,986 | new=1,500 | reused=0
Hashed/checkpointed: 1,600 / 2,986 |

In [17]:
# ============================================================
# CELL 10 — BUILD FROZEN DENSENET121 MODEL
# ============================================================

import os
import json
from pathlib import Path

import tensorflow as tf
import yaml

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dropout,
    Dense
)
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import (
    BinaryAccuracy,
    Precision,
    Recall,
    AUC
)

print("\n" + "=" * 85)
print("BUILDING FROZEN DENSENET121 MODEL")
print("=" * 85)


# ------------------------------------------------------------
# Mixed precision for T4 GPU
# ------------------------------------------------------------
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy(
    "mixed_float16"
)

print(
    "Mixed precision policy:",
    mixed_precision.global_policy()
)


# ------------------------------------------------------------
# Model hyperparameters
# ------------------------------------------------------------
MODEL_CONFIG = {
    "model_name": "DenseNet121",
    "weights": "imagenet",
    "include_top": False,
    "input_shape": [
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3
    ],
    "global_pooling": (
        "GlobalAveragePooling2D"
    ),
    "dropout_rate": 0.40,
    "output_units": 1,
    "output_activation": "sigmoid",
    "output_dtype": "float32",
    "frozen_learning_rate": 1e-3,
    "finetune_learning_rate": 1e-5,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "label_mapping": {
        "real": 0,
        "fake": 1
    },
    "model_selection_metric": "val_auc",
    "model_selection_mode": "max"
}


# ------------------------------------------------------------
# Build ImageNet-pretrained DenseNet121 backbone
# ------------------------------------------------------------
print(
    "\nLoading ImageNet-pretrained "
    "DenseNet121 weights..."
)

base_model = DenseNet121(
    weights=MODEL_CONFIG["weights"],
    include_top=MODEL_CONFIG["include_top"],
    input_shape=tuple(
        MODEL_CONFIG["input_shape"]
    )
)

# Freeze the complete DenseNet121 backbone
base_model.trainable = False


# ------------------------------------------------------------
# Build binary classification model
# ------------------------------------------------------------
model_input = Input(
    shape=tuple(
        MODEL_CONFIG["input_shape"]
    ),
    name="eye_roi_input"
)

# training=False keeps Batch Normalization layers
# in inference mode during frozen training.
features = base_model(
    model_input,
    training=False
)

features = GlobalAveragePooling2D(
    name="global_average_pooling"
)(features)

features = Dropout(
    rate=MODEL_CONFIG["dropout_rate"],
    seed=SEED,
    name="classification_dropout"
)(features)

model_output = Dense(
    units=MODEL_CONFIG["output_units"],
    activation=MODEL_CONFIG[
        "output_activation"
    ],
    dtype=MODEL_CONFIG[
        "output_dtype"
    ],
    name="real_fake_probability"
)(features)

model = Model(
    inputs=model_input,
    outputs=model_output,
    name="DenseNet121_Eye_Deepfake"
)


# ------------------------------------------------------------
# Compile frozen-stage model
# ------------------------------------------------------------
frozen_optimizer = Adam(
    learning_rate=MODEL_CONFIG[
        "frozen_learning_rate"
    ]
)

model.compile(
    optimizer=frozen_optimizer,

    loss=BinaryCrossentropy(
        name="binary_crossentropy"
    ),

    metrics=[
        BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),

        Precision(
            name="precision",
            thresholds=0.5
        ),

        Recall(
            name="recall",
            thresholds=0.5
        ),

        AUC(
            name="auc",
            curve="ROC"
        ),

        AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


# ------------------------------------------------------------
# Parameter accounting
# ------------------------------------------------------------
total_parameters = model.count_params()

trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in model.trainable_weights
    )
)

non_trainable_parameters = (
    total_parameters -
    trainable_parameters
)

base_total_parameters = (
    base_model.count_params()
)

base_trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in base_model.trainable_weights
    )
)


# ------------------------------------------------------------
# Model assertions
# ------------------------------------------------------------
assert base_model.trainable is False, (
    "DenseNet121 backbone must be frozen."
)

assert base_trainable_parameters == 0, (
    "Frozen DenseNet121 contains trainable parameters."
)

assert model.output_shape == (
    None,
    1
), (
    f"Unexpected model output shape: "
    f"{model.output_shape}"
)

assert model.output.dtype == "float32", (
    "The sigmoid output must use float32 "
    "under mixed precision."
)


# ------------------------------------------------------------
# Forward-pass smoke check
# No model training is performed here.
# ------------------------------------------------------------
forward_test_batch = sample_images[:2]

forward_test_predictions = model(
    forward_test_batch,
    training=False
)

assert forward_test_predictions.shape == (
    2,
    1
), (
    "Forward-pass output shape is incorrect."
)

assert bool(
    tf.reduce_all(
        tf.math.is_finite(
            forward_test_predictions
        )
    )
), (
    "NaN or Inf was detected in model outputs."
)

assert bool(
    tf.reduce_all(
        forward_test_predictions >= 0.0
    )
), (
    "Prediction below zero was detected."
)

assert bool(
    tf.reduce_all(
        forward_test_predictions <= 1.0
    )
), (
    "Prediction above one was detected."
)


# ------------------------------------------------------------
# Standard output paths
# ------------------------------------------------------------
BEST_MODEL_PATH = (
    CHECKPOINT_DIR /
    "best.keras"
)

LAST_MODEL_PATH = (
    CHECKPOINT_DIR /
    "last.keras"
)

BACKUP_DIR = (
    CHECKPOINT_DIR /
    "training_backup"
)

MODEL_SUMMARY_PATH = (
    METRICS_DIR /
    "model_summary.txt"
)

RESOLVED_CONFIG_PATH = (
    RUN_ROOT /
    "config_resolved.yaml"
)


# ------------------------------------------------------------
# Save model summary
# ------------------------------------------------------------
summary_lines = []

model.summary(
    print_fn=lambda line: (
        summary_lines.append(line)
    )
)

with open(
    MODEL_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as summary_file:
    summary_file.write(
        "\n".join(summary_lines)
    )


# ------------------------------------------------------------
# Save resolved experiment configuration atomically
# ------------------------------------------------------------
resolved_config = {
    "run_id": RUN_ID,

    "task": (
        "Eye ROI Deepfake "
        "Binary Classification"
    ),

    "framework": "TensorFlow / Keras",

    "tensorflow_version": tf.__version__,

    "mixed_precision_policy": str(
        mixed_precision.global_policy()
    ),

    "dataset": {
        "dataset_root": str(DATASET_ROOT),
        "manifest_path": str(
            MANIFEST_PATH
        ),
        "train_count": actual_counts["train"]["total"],
        "validation_count": actual_counts["val"]["total"],
        "test_count": actual_counts["test"]["total"],
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "channels": 3
    },

    "model": MODEL_CONFIG,

    "parameters": {
        "total": total_parameters,
        "trainable_frozen_stage": (
            trainable_parameters
        ),
        "non_trainable_frozen_stage": (
            non_trainable_parameters
        ),
        "base_model_total": (
            base_total_parameters
        ),
        "base_model_trainable": (
            base_trainable_parameters
        )
    },

    "output_paths": {
        "run_root": str(RUN_ROOT),
        "best_model": str(
            BEST_MODEL_PATH
        ),
        "last_model": str(
            LAST_MODEL_PATH
        ),
        "backup_directory": str(
            BACKUP_DIR
        )
    }
}

temporary_config_path = (
    RESOLVED_CONFIG_PATH.with_suffix(
        ".yaml.tmp"
    )
)

with open(
    temporary_config_path,
    "w",
    encoding="utf-8"
) as config_file:
    yaml.safe_dump(
        resolved_config,
        config_file,
        allow_unicode=True,
        sort_keys=False
    )

# Atomic replacement
os.replace(
    temporary_config_path,
    RESOLVED_CONFIG_PATH
)


# ------------------------------------------------------------
# Print model information
# ------------------------------------------------------------
print("\n" + "-" * 85)
print("MODEL PARAMETER SUMMARY")
print("-" * 85)

print(
    f"Total parameters        : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters    : "
    f"{trainable_parameters:,}"
)

print(
    f"Non-trainable parameters: "
    f"{non_trainable_parameters:,}"
)

print(
    f"Backbone parameters     : "
    f"{base_total_parameters:,}"
)

print(
    f"Backbone trainable      : "
    f"{base_trainable_parameters:,}"
)

print("\nForward-test predictions:")

print(
    forward_test_predictions
    .numpy()
    .reshape(-1)
    .tolist()
)

print("\nModel summary :", MODEL_SUMMARY_PATH)
print("Config file  :", RESOLVED_CONFIG_PATH)
print("Best model   :", BEST_MODEL_PATH)
print("Last model   :", LAST_MODEL_PATH)

print("=" * 85)
print("DENSENET121 BACKBONE     : FROZEN")
print("MIXED PRECISION           : ENABLED")
print("FORWARD PASS TEST         : PASS")
print("MODEL CONFIGURATION       : SAVED")
print("CELL 10 COMPLETED")
print("=" * 85)



BUILDING FROZEN DENSENET121 MODEL
Mixed precision policy: <DTypePolicy "mixed_float16">

Loading ImageNet-pretrained DenseNet121 weights...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step



-------------------------------------------------------------------------------------
MODEL PARAMETER SUMMARY
-------------------------------------------------------------------------------------
Total parameters        : 7,038,529
Trainable parameters    : 1,025
Non-trainable parameters: 7,037,504
Backbone parameters     : 7,037,504
Backbone trainable      : 0

Forward-test predictions:
[0.3582615554332733, 0.39686742424964905]

Model summary : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/metrics/model_summary.txt
Config file  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/config_resolved.yaml
Best model   : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/best.keras
Last model   : /content

In [18]:
# ============================================================
# CELL 11 — TWO-BATCH TRAINING AND CHECKPOINT SMOKE TEST
# ============================================================

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

print("\n" + "=" * 90)
print("TWO-BATCH TRAINING AND CHECKPOINT SMOKE TEST")
print("=" * 90)


# ------------------------------------------------------------
# Smoke-test checkpoint paths
# ------------------------------------------------------------
SMOKE_CHECKPOINT_PATH = (
    CHECKPOINT_DIR /
    "smoke_test_checkpoint.keras"
)

SMOKE_TEMP_PATH = (
    CHECKPOINT_DIR /
    "smoke_test_checkpoint.tmp.keras"
)


# ------------------------------------------------------------
# Keep the original untrained classification-head weights
# The smoke test must not affect the real experiment.
# ------------------------------------------------------------
initial_model_weights = model.get_weights()

initial_optimizer_iterations = int(
    model.optimizer.iterations.numpy()
)

print(
    "Initial optimizer iterations:",
    initial_optimizer_iterations
)


# ------------------------------------------------------------
# Obtain exactly two training batches
# ------------------------------------------------------------
smoke_batches = list(
    train_dataset.take(2)
)

assert len(smoke_batches) == 2, (
    "Two training batches could not be loaded."
)

for batch_index, (
    batch_images,
    batch_labels
) in enumerate(
    smoke_batches,
    start=1
):
    print(
        f"Batch {batch_index} shape:",
        batch_images.shape,
        batch_labels.shape
    )

    assert batch_images.shape[1:] == (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        3
    )

    assert bool(
        tf.reduce_all(
            tf.math.is_finite(
                batch_images
            )
        )
    ), (
        f"NaN or Inf found in batch "
        f"{batch_index} images."
    )

    assert bool(
        tf.reduce_all(
            tf.math.is_finite(
                batch_labels
            )
        )
    ), (
        f"NaN or Inf found in batch "
        f"{batch_index} labels."
    )


# ------------------------------------------------------------
# Explicit forward and gradient numerical test
# No optimizer update is performed in this section.
# ------------------------------------------------------------
gradient_test_images = smoke_batches[0][0]
gradient_test_labels = smoke_batches[0][1]

gradient_test_labels = tf.reshape(
    gradient_test_labels,
    shape=(-1, 1)
)

gradient_loss_function = (
    tf.keras.losses.BinaryCrossentropy()
)

with tf.GradientTape() as tape:

    gradient_predictions = model(
        gradient_test_images,
        training=True
    )

    gradient_loss = gradient_loss_function(
        gradient_test_labels,
        gradient_predictions
    )

gradients = tape.gradient(
    gradient_loss,
    model.trainable_variables
)

assert bool(
    tf.math.is_finite(
        gradient_loss
    )
), (
    "NaN or Inf detected in smoke-test loss."
)

assert gradients, (
    "No gradients were generated."
)

assert all(
    gradient is not None
    for gradient in gradients
), (
    "A trainable variable has a missing gradient."
)

for gradient_index, gradient in enumerate(
    gradients
):
    assert bool(
        tf.reduce_all(
            tf.math.is_finite(
                gradient
            )
        )
    ), (
        f"NaN or Inf detected in gradient "
        f"{gradient_index}."
    )

print(
    "\nExplicit gradient-test loss:",
    float(gradient_loss.numpy())
)

print(
    "Gradient tensors checked:",
    len(gradients)
)


# ------------------------------------------------------------
# Perform exactly two optimizer update steps
# ------------------------------------------------------------
training_step_results = []

for batch_index, (
    batch_images,
    batch_labels
) in enumerate(
    smoke_batches,
    start=1
):

    batch_result = model.train_on_batch(
        batch_images,
        batch_labels,
        return_dict=True
    )

    clean_batch_result = {
        metric_name: float(metric_value)
        for metric_name, metric_value
        in batch_result.items()
    }

    assert all(
        np.isfinite(metric_value)
        for metric_value
        in clean_batch_result.values()
    ), (
        f"NaN or Inf detected after "
        f"training batch {batch_index}."
    )

    training_step_results.append(
        clean_batch_result
    )

    print(
        f"\nTraining batch {batch_index}:"
    )

    for metric_name, metric_value in (
        clean_batch_result.items()
    ):
        print(
            f"  {metric_name:12}: "
            f"{metric_value:.6f}"
        )


optimizer_iterations_after_training = int(
    model.optimizer.iterations.numpy()
)

assert optimizer_iterations_after_training == (
    initial_optimizer_iterations + 2
), (
    "Optimizer iteration count did not "
    "increase by exactly two."
)


# ------------------------------------------------------------
# Reference predictions and loss before saving
# Use validation data without augmentation.
# ------------------------------------------------------------
reference_images, reference_labels = next(
    iter(val_dataset)
)

reference_labels_2d = tf.reshape(
    reference_labels,
    shape=(-1, 1)
)

predictions_before_save = model(
    reference_images,
    training=False
).numpy()

loss_before_save = float(
    gradient_loss_function(
        reference_labels_2d,
        predictions_before_save
    ).numpy()
)

assert np.isfinite(
    predictions_before_save
).all()

assert np.isfinite(
    loss_before_save
)


# ------------------------------------------------------------
# Full-state atomic checkpoint save
# The .keras file stores architecture, weights and optimizer.
# ------------------------------------------------------------
if SMOKE_TEMP_PATH.exists():
    SMOKE_TEMP_PATH.unlink()

model.save(
    SMOKE_TEMP_PATH,
    include_optimizer=True
)

# Verify the temporary checkpoint before publishing it
temporary_loaded_model = (
    tf.keras.models.load_model(
        SMOKE_TEMP_PATH
    )
)

temporary_predictions = (
    temporary_loaded_model(
        reference_images,
        training=False
    )
    .numpy()
)

assert np.allclose(
    predictions_before_save,
    temporary_predictions,
    rtol=1e-6,
    atol=1e-7
), (
    "Temporary checkpoint prediction "
    "verification failed."
)

# Atomic publish
os.replace(
    SMOKE_TEMP_PATH,
    SMOKE_CHECKPOINT_PATH
)


# ------------------------------------------------------------
# Reload the published checkpoint from disk
# ------------------------------------------------------------
reloaded_smoke_model = (
    tf.keras.models.load_model(
        SMOKE_CHECKPOINT_PATH
    )
)

predictions_after_reload = (
    reloaded_smoke_model(
        reference_images,
        training=False
    )
    .numpy()
)

loss_after_reload = float(
    gradient_loss_function(
        reference_labels_2d,
        predictions_after_reload
    ).numpy()
)

reloaded_optimizer_iterations = int(
    reloaded_smoke_model
    .optimizer
    .iterations
    .numpy()
)


# ------------------------------------------------------------
# Checkpoint equivalence assertions
# ------------------------------------------------------------
assert np.allclose(
    predictions_before_save,
    predictions_after_reload,
    rtol=1e-6,
    atol=1e-7
), (
    "Predictions changed after "
    "checkpoint reload."
)

assert np.isclose(
    loss_before_save,
    loss_after_reload,
    rtol=1e-6,
    atol=1e-7
), (
    "Loss changed after checkpoint reload."
)

assert reloaded_optimizer_iterations == (
    optimizer_iterations_after_training
), (
    "Optimizer state was not restored."
)


# ------------------------------------------------------------
# Restore the original pre-smoke-test weights
# ------------------------------------------------------------
model.set_weights(
    initial_model_weights
)

# Reset the optimizer so the real training starts cleanly.
frozen_optimizer = tf.keras.optimizers.Adam(
    learning_rate=MODEL_CONFIG[
        "frozen_learning_rate"
    ]
)

model.compile(
    optimizer=frozen_optimizer,

    loss=tf.keras.losses.BinaryCrossentropy(
        name="binary_crossentropy"
    ),

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),

        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),

        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),

        tf.keras.metrics.AUC(
            name="auc",
            curve="ROC"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

assert int(
    model.optimizer.iterations.numpy()
) == 0, (
    "The real-training optimizer did not "
    "restart from iteration zero."
)


# ------------------------------------------------------------
# Verify restored initial predictions are finite
# ------------------------------------------------------------
restored_predictions = model(
    reference_images,
    training=False
).numpy()

assert np.isfinite(
    restored_predictions
).all(), (
    "Restored model produced NaN or Inf."
)


# ------------------------------------------------------------
# Save smoke-test report
# ------------------------------------------------------------
SMOKE_REPORT_PATH = (
    METRICS_DIR /
    "smoke_test_report.json"
)

smoke_test_report = {
    "run_id": RUN_ID,
    "tested_batches": 2,
    "gradient_loss": float(
        gradient_loss.numpy()
    ),
    "gradient_tensor_count": len(
        gradients
    ),
    "training_step_results": (
        training_step_results
    ),
    "optimizer_iterations_before": (
        initial_optimizer_iterations
    ),
    "optimizer_iterations_after_two_batches": (
        optimizer_iterations_after_training
    ),
    "reloaded_optimizer_iterations": (
        reloaded_optimizer_iterations
    ),
    "loss_before_save": (
        loss_before_save
    ),
    "loss_after_reload": (
        loss_after_reload
    ),
    "prediction_max_absolute_difference": (
        float(
            np.max(
                np.abs(
                    predictions_before_save -
                    predictions_after_reload
                )
            )
        )
    ),
    "checkpoint_path": str(
        SMOKE_CHECKPOINT_PATH
    ),
    "real_training_optimizer_reset": True,
    "status": "PASS"
}

temporary_report_path = (
    SMOKE_REPORT_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_report_path,
    "w",
    encoding="utf-8"
) as report_file:
    json.dump(
        smoke_test_report,
        report_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_report_path,
    SMOKE_REPORT_PATH
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("\n" + "-" * 90)
print("CHECKPOINT RELOAD RESULTS")
print("-" * 90)

print(
    "Loss before save       :",
    loss_before_save
)

print(
    "Loss after reload      :",
    loss_after_reload
)

print(
    "Maximum prediction diff:",
    smoke_test_report[
        "prediction_max_absolute_difference"
    ]
)

print(
    "Optimizer iterations   :",
    reloaded_optimizer_iterations
)

print(
    "Real optimizer reset   :",
    int(
        model.optimizer
        .iterations
        .numpy()
    )
)

print("Checkpoint :", SMOKE_CHECKPOINT_PATH)
print("Report     :", SMOKE_REPORT_PATH)

print("=" * 90)
print("TWO-BATCH TRAINING TEST : PASS")
print("GRADIENT NaN/Inf TEST   : PASS")
print("ATOMIC CHECKPOINT TEST  : PASS")
print("CHECKPOINT RELOAD TEST  : PASS")
print("REAL MODEL RESET        : PASS")
print("CELL 11 COMPLETED")
print("=" * 90)



TWO-BATCH TRAINING AND CHECKPOINT SMOKE TEST
Initial optimizer iterations: 0
Batch 1 shape: (16, 224, 224, 3) (16,)
Batch 2 shape: (16, 224, 224, 3) (16,)

Explicit gradient-test loss: 0.5642508268356323
Gradient tensors checked: 2

Training batch 1:
  accuracy    : 0.625000
  auc         : 0.539683
  loss        : 0.820895
  pr_auc      : 0.552957
  precision   : 0.714286
  recall      : 0.555556

Training batch 2:
  accuracy    : 0.593750
  auc         : 0.556863
  loss        : 0.769376
  pr_auc      : 0.571768
  precision   : 0.642857
  recall      : 0.529412

------------------------------------------------------------------------------------------
CHECKPOINT RELOAD RESULTS
------------------------------------------------------------------------------------------
Loss before save       : 0.8206490278244019
Loss after reload      : 0.8206490278244019
Maximum prediction diff: 0.0
Optimizer iterations   : 2
Real optimizer reset   : 0
Checkpoint : /content/drive/MyDrive/AISC DeepFake

In [19]:
# ============================================================
# CELL 12 — FROZEN-BACKBONE TRAINING
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print("\n" + "=" * 90)
print("FROZEN-BACKBONE TRAINING")
print("=" * 90)


# ------------------------------------------------------------
# Frozen-stage configuration
# ------------------------------------------------------------
FROZEN_EPOCHS = int(CONFIG["frozen_epochs"])

FROZEN_HISTORY_PATH = (
    METRICS_DIR /
    "frozen_training_history.csv"
)

FROZEN_LOG_PATH = (
    LOG_DIR /
    "frozen_training_log.csv"
)

CHECKPOINT_STATE_PATH = (
    CHECKPOINT_DIR /
    "checkpoint_state.json"
)


# ------------------------------------------------------------
# Atomic full-model checkpoint callback
# ------------------------------------------------------------
class AtomicModelCheckpoint(
    tf.keras.callbacks.Callback
):
    """
    Saves:
    - last.keras after every completed epoch
    - best.keras when validation AUC improves

    Each model is first written to a temporary file,
    reloaded for integrity verification and then
    atomically published with os.replace().
    """

    def __init__(
        self,
        best_model_path,
        last_model_path,
        state_path,
        monitor="val_auc",
        mode="max"
    ):
        super().__init__()

        self.best_model_path = Path(
            best_model_path
        )

        self.last_model_path = Path(
            last_model_path
        )

        self.state_path = Path(
            state_path
        )

        self.monitor = monitor
        self.mode = mode

        if mode == "max":
            self.best_value = -np.inf
        elif mode == "min":
            self.best_value = np.inf
        else:
            raise ValueError(
                "mode must be 'max' or 'min'."
            )

        # Recover the previous best value if this
        # experiment is resumed.
        if self.state_path.exists():

            with open(
                self.state_path,
                "r",
                encoding="utf-8"
            ) as state_file:
                previous_state = json.load(
                    state_file
                )

            if self.monitor in previous_state:
                self.best_value = float(
                    previous_state[
                        self.monitor
                    ]
                )

    def _is_improvement(self, current_value):

        if self.mode == "max":
            return current_value > self.best_value

        return current_value < self.best_value

    def _atomic_save_model(
        self,
        target_path
    ):
        target_path = Path(target_path)

        temporary_path = (
            target_path.parent /
            f"{target_path.stem}.tmp.keras"
        )

        if temporary_path.exists():
            temporary_path.unlink()

        # Save complete model including optimizer state
        self.model.save(
            temporary_path,
            include_optimizer=True
        )

        # Integrity verification
        verification_model = (
            tf.keras.models.load_model(
                temporary_path
            )
        )

        assert verification_model.output_shape == (
            None,
            1
        ), (
            "Checkpoint verification failed: "
            "unexpected output shape."
        )

        # Release temporary verification model
        del verification_model

        # Atomic publish
        os.replace(
            temporary_path,
            target_path
        )

    def _atomic_save_state(
        self,
        epoch,
        current_value
    ):
        state_data = {
            "run_id": RUN_ID,
            "completed_epoch": int(
                epoch + 1
            ),
            "monitor": self.monitor,
            self.monitor: float(
                self.best_value
            ),
            "latest_value": float(
                current_value
            ),
            "best_model_path": str(
                self.best_model_path
            ),
            "last_model_path": str(
                self.last_model_path
            )
        }

        temporary_state_path = (
            self.state_path.with_suffix(
                ".json.tmp"
            )
        )

        with open(
            temporary_state_path,
            "w",
            encoding="utf-8"
        ) as state_file:
            json.dump(
                state_data,
                state_file,
                indent=4,
                ensure_ascii=False
            )

        os.replace(
            temporary_state_path,
            self.state_path
        )

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):
        logs = logs or {}

        if self.monitor not in logs:
            raise KeyError(
                f"Monitored metric not found: "
                f"{self.monitor}"
            )

        current_value = float(
            logs[self.monitor]
        )

        if not np.isfinite(current_value):
            raise FloatingPointError(
                f"{self.monitor} is NaN or Inf."
            )

        # Always save the most recently completed epoch
        self._atomic_save_model(
            self.last_model_path
        )

        improved = self._is_improvement(
            current_value
        )

        if improved:
            self.best_value = current_value

            self._atomic_save_model(
                self.best_model_path
            )

            print(
                f"\nAtomic best checkpoint updated: "
                f"{self.monitor}="
                f"{current_value:.6f}"
            )

        self._atomic_save_state(
            epoch=epoch,
            current_value=current_value
        )


# ------------------------------------------------------------
# Epoch-level finite-value quality gate
# ------------------------------------------------------------
class FiniteMetricsGuard(
    tf.keras.callbacks.Callback
):
    """
    Stops training immediately if an epoch metric
    contains NaN or Inf.
    """

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):
        logs = logs or {}

        invalid_metrics = {
            metric_name: metric_value
            for metric_name, metric_value
            in logs.items()
            if not np.isfinite(metric_value)
        }

        if invalid_metrics:
            raise FloatingPointError(
                "NaN or Inf detected in metrics: "
                f"{invalid_metrics}"
            )


# ------------------------------------------------------------
# Training callbacks
# ------------------------------------------------------------
atomic_checkpoint_callback = (
    AtomicModelCheckpoint(
        best_model_path=BEST_MODEL_PATH,
        last_model_path=LAST_MODEL_PATH,
        state_path=CHECKPOINT_STATE_PATH,
        monitor="val_auc",
        mode="max"
    )
)

frozen_callbacks = [
    atomic_checkpoint_callback,

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(BACKUP_DIR),
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(FROZEN_LOG_PATH),
        separator=",",
        append=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        min_delta=1e-4,
        restore_best_weights=False,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN(),

    FiniteMetricsGuard()
]


# ------------------------------------------------------------
# Pre-training assertions
# ------------------------------------------------------------
assert base_model.trainable is False, (
    "DenseNet121 must remain frozen "
    "during stage 1."
)

assert int(
    model.optimizer.iterations.numpy()
) == 0, (
    "Frozen training must start from "
    "optimizer iteration zero."
)

frozen_trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in model.trainable_weights
    )
)

assert frozen_trainable_parameters == 1025, (
    "Unexpected frozen-stage trainable "
    f"parameter count: "
    f"{frozen_trainable_parameters}"
)


# ------------------------------------------------------------
# Start frozen-backbone training
# ------------------------------------------------------------
print(
    "Run ID                 :",
    RUN_ID
)

print(
    "Frozen epochs maximum  :",
    FROZEN_EPOCHS
)

print(
    "Trainable parameters   :",
    f"{frozen_trainable_parameters:,}"
)

print(
    "Selection metric       : val_auc"
)

print(
    "Best checkpoint        :",
    BEST_MODEL_PATH
)

print(
    "Last checkpoint        :",
    LAST_MODEL_PATH
)

print(
    "Recovery backup        :",
    BACKUP_DIR
)

print(
    "\nStarting frozen-backbone "
    "training...\n"
)

frozen_history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=FROZEN_EPOCHS,
    callbacks=frozen_callbacks,
    verbose=1
)


# ------------------------------------------------------------
# Convert training history to a table
# ------------------------------------------------------------
frozen_history_df = pd.DataFrame(
    frozen_history.history
)

frozen_history_df.insert(
    0,
    "epoch",
    np.arange(
        1,
        len(frozen_history_df) + 1
    )
)

frozen_history_df.insert(
    1,
    "stage",
    "frozen"
)


# ------------------------------------------------------------
# Save history atomically
# ------------------------------------------------------------
temporary_history_path = (
    FROZEN_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )
)

frozen_history_df.to_csv(
    temporary_history_path,
    index=False
)

os.replace(
    temporary_history_path,
    FROZEN_HISTORY_PATH
)


# ------------------------------------------------------------
# Verify checkpoints
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "Best model checkpoint was not created."
)

assert LAST_MODEL_PATH.exists(), (
    "Last model checkpoint was not created."
)

verified_best_model = (
    tf.keras.models.load_model(
        BEST_MODEL_PATH
    )
)

verified_last_model = (
    tf.keras.models.load_model(
        LAST_MODEL_PATH
    )
)

best_check_predictions = (
    verified_best_model(
        sample_images[:2],
        training=False
    )
    .numpy()
)

last_check_predictions = (
    verified_last_model(
        sample_images[:2],
        training=False
    )
    .numpy()
)

assert np.isfinite(
    best_check_predictions
).all()

assert np.isfinite(
    last_check_predictions
).all()


# ------------------------------------------------------------
# Frozen-stage summary
# ------------------------------------------------------------
best_frozen_epoch_index = int(
    frozen_history_df[
        "val_auc"
    ].idxmax()
)

best_frozen_row = (
    frozen_history_df.loc[
        best_frozen_epoch_index
    ]
)

FROZEN_SUMMARY_PATH = (
    METRICS_DIR /
    "frozen_training_summary.json"
)

frozen_summary = {
    "run_id": RUN_ID,
    "stage": "frozen",
    "epochs_completed": int(
        len(frozen_history_df)
    ),
    "maximum_epochs": FROZEN_EPOCHS,
    "trainable_parameters": (
        frozen_trainable_parameters
    ),
    "best_epoch_in_this_stage": int(
        best_frozen_row["epoch"]
    ),
    "best_val_auc_in_this_stage": float(
        best_frozen_row["val_auc"]
    ),
    "best_val_loss_in_this_stage": float(
        frozen_history_df[
            "val_loss"
        ].min()
    ),
    "best_model_path": str(
        BEST_MODEL_PATH
    ),
    "last_model_path": str(
        LAST_MODEL_PATH
    )
}

temporary_summary_path = (
    FROZEN_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_summary_path,
    "w",
    encoding="utf-8"
) as summary_file:
    json.dump(
        frozen_summary,
        summary_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_summary_path,
    FROZEN_SUMMARY_PATH
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("FROZEN TRAINING COMPLETED")
print("=" * 90)

print(
    "Epochs completed       :",
    len(frozen_history_df)
)

print(
    "Best frozen epoch      :",
    int(best_frozen_row["epoch"])
)

print(
    "Best frozen val AUC    :",
    float(best_frozen_row["val_auc"])
)

print(
    "Minimum frozen val loss:",
    float(
        frozen_history_df[
            "val_loss"
        ].min()
    )
)

print(
    "Best checkpoint        :",
    BEST_MODEL_PATH
)

print(
    "Last checkpoint        :",
    LAST_MODEL_PATH
)

print(
    "History CSV            :",
    FROZEN_HISTORY_PATH
)

print(
    "Summary JSON           :",
    FROZEN_SUMMARY_PATH
)

print("=" * 90)
print("FROZEN TRAINING          : PASS")
print("BEST CHECKPOINT          : VERIFIED")
print("LAST CHECKPOINT          : VERIFIED")
print("RECOVERY BACKUP          : ENABLED")
print("CELL 12 COMPLETED")
print("=" * 90)



FROZEN-BACKBONE TRAINING
Run ID                 : 20260808_1214_eye_densenet121_seed42
Frozen epochs maximum  : 12
Trainable parameters   : 1,025
Selection metric       : val_auc
Best checkpoint        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/best.keras
Last checkpoint        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/last.keras
Recovery backup        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/training_backup

Starting frozen-backbone training...

Epoch 1/12
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 455ms/step - accuracy: 0.5101 - auc: 0.5274 - loss: 0.7495 - pr_auc: 0.5232 - precision: 0.5128 - recall: 0.5287
Atomic best checkpoint updated: val_auc=0.630634
1

In [20]:
# ============================================================
# CELL 13 — PARTIAL DENSENET121 FINE-TUNING
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print("\n" + "=" * 90)
print("PARTIAL DENSENET121 FINE-TUNING")
print("=" * 90)


# ------------------------------------------------------------
# Fine-tuning configuration
# ------------------------------------------------------------
FINETUNE_EPOCHS = int(CONFIG["finetune_epochs"])
FROZEN_EPOCHS_COMPLETED = int(
    frozen_summary["epochs_completed"]
)

FINETUNE_INITIAL_EPOCH = (
    FROZEN_EPOCHS_COMPLETED
)

FINETUNE_FINAL_EPOCH = (
    FINETUNE_INITIAL_EPOCH +
    FINETUNE_EPOCHS
)

FINETUNE_HISTORY_PATH = (
    METRICS_DIR /
    "finetune_training_history.csv"
)

FINETUNE_LOG_PATH = (
    LOG_DIR /
    "finetune_training_log.csv"
)

FINETUNE_SUMMARY_PATH = (
    METRICS_DIR /
    "finetune_training_summary.json"
)

FINETUNE_BACKUP_DIR = (
    CHECKPOINT_DIR /
    "finetune_training_backup"
)


# ------------------------------------------------------------
# Load the best frozen-stage model
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "The best frozen-stage checkpoint "
    "does not exist."
)

model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

print(
    "Best frozen checkpoint loaded:",
    BEST_MODEL_PATH
)


# ------------------------------------------------------------
# Locate the DenseNet121 backbone
# ------------------------------------------------------------
base_model = model.get_layer(
    "densenet121"
)

# Enable layer-specific fine-tuning
base_model.trainable = True


# ------------------------------------------------------------
# Freeze every backbone layer first
# ------------------------------------------------------------
for layer in base_model.layers:
    layer.trainable = False


# ------------------------------------------------------------
# Unfreeze only the final DenseNet block
#
# DenseNet121 final block:
# conv5_block1 ... conv5_block16
#
# Batch Normalization layers remain frozen.
# ------------------------------------------------------------
unfrozen_layer_names = []
frozen_batch_norm_names = []

for layer in base_model.layers:

    is_final_dense_block = (
        layer.name.startswith(
            "conv5_block"
        )
    )

    is_batch_normalization = isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    )

    if (
        is_final_dense_block
        and not is_batch_normalization
    ):
        layer.trainable = True

        unfrozen_layer_names.append(
            layer.name
        )

    elif (
        is_final_dense_block
        and is_batch_normalization
    ):
        layer.trainable = False

        frozen_batch_norm_names.append(
            layer.name
        )


# ------------------------------------------------------------
# Layer-level quality assertions
# ------------------------------------------------------------
assert unfrozen_layer_names, (
    "No DenseNet121 final-block layers "
    "were unfrozen."
)

assert all(
    not layer.trainable
    for layer in base_model.layers
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    )
), (
    "A Batch Normalization layer was "
    "incorrectly unfrozen."
)

assert all(
    not layer.trainable
    for layer in base_model.layers
    if (
        not layer.name.startswith(
            "conv5_block"
        )
        and not isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        )
    )
), (
    "A layer outside the final dense block "
    "was incorrectly unfrozen."
)


# ------------------------------------------------------------
# Compile with a low fine-tuning learning rate
# ------------------------------------------------------------
finetune_optimizer = (
    tf.keras.optimizers.Adam(
        learning_rate=MODEL_CONFIG[
            "finetune_learning_rate"
        ]
    )
)

model.compile(
    optimizer=finetune_optimizer,

    loss=tf.keras.losses.BinaryCrossentropy(
        name="binary_crossentropy"
    ),

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy",
            threshold=0.5
        ),

        tf.keras.metrics.Precision(
            name="precision",
            thresholds=0.5
        ),

        tf.keras.metrics.Recall(
            name="recall",
            thresholds=0.5
        ),

        tf.keras.metrics.AUC(
            name="auc",
            curve="ROC"
        ),

        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)


# ------------------------------------------------------------
# Parameter accounting
# ------------------------------------------------------------
finetune_total_parameters = (
    model.count_params()
)

finetune_trainable_parameters = int(
    sum(
        tf.keras.backend.count_params(
            variable
        )
        for variable in model.trainable_weights
    )
)

finetune_non_trainable_parameters = (
    finetune_total_parameters -
    finetune_trainable_parameters
)

assert finetune_trainable_parameters > 1025, (
    "Fine-tuning did not add trainable "
    "backbone parameters."
)

assert finetune_trainable_parameters < (
    finetune_total_parameters
), (
    "The complete DenseNet121 model was "
    "accidentally unfrozen."
)


# ------------------------------------------------------------
# Forward-pass numerical quality check
# ------------------------------------------------------------
finetune_test_predictions = model(
    sample_images[:2],
    training=False
).numpy()

assert np.isfinite(
    finetune_test_predictions
).all(), (
    "Fine-tuning model produced NaN or Inf."
)


# ------------------------------------------------------------
# Fine-tuning callbacks
# ------------------------------------------------------------
finetune_atomic_checkpoint = (
    AtomicModelCheckpoint(
        best_model_path=BEST_MODEL_PATH,
        last_model_path=LAST_MODEL_PATH,
        state_path=CHECKPOINT_STATE_PATH,
        monitor="val_auc",
        mode="max"
    )
)

finetune_callbacks = [
    finetune_atomic_checkpoint,

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(
            FINETUNE_BACKUP_DIR
        ),
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(
            FINETUNE_LOG_PATH
        ),
        separator=",",
        append=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=4,
        min_delta=1e-4,
        restore_best_weights=False,
        verbose=1
    ),

    tf.keras.callbacks.TerminateOnNaN(),

    FiniteMetricsGuard()
]


# ------------------------------------------------------------
# Print fine-tuning setup
# ------------------------------------------------------------
print("\n" + "-" * 90)
print("FINE-TUNING CONFIGURATION")
print("-" * 90)

print(
    "Initial checkpoint       :",
    BEST_MODEL_PATH
)

print(
    "Fine-tuning learning rate:",
    MODEL_CONFIG[
        "finetune_learning_rate"
    ]
)

print(
    "Maximum fine-tune epochs :",
    FINETUNE_EPOCHS
)

print(
    "Epoch numbering          :",
    f"{FINETUNE_INITIAL_EPOCH + 1} "
    f"to {FINETUNE_FINAL_EPOCH}"
)

print(
    "Unfrozen backbone layers :",
    len(unfrozen_layer_names)
)

print(
    "Frozen BatchNorm layers  :",
    len(frozen_batch_norm_names)
)

print(
    "Total parameters         :",
    f"{finetune_total_parameters:,}"
)

print(
    "Trainable parameters     :",
    f"{finetune_trainable_parameters:,}"
)

print(
    "Non-trainable parameters :",
    f"{finetune_non_trainable_parameters:,}"
)

print("\nFirst five unfrozen layers:")

for layer_name in unfrozen_layer_names[:5]:
    print(" -", layer_name)

print("\nLast five unfrozen layers:")

for layer_name in unfrozen_layer_names[-5:]:
    print(" -", layer_name)


# ------------------------------------------------------------
# Start partial fine-tuning
# ------------------------------------------------------------
print(
    "\nStarting partial "
    "fine-tuning...\n"
)

finetune_history = model.fit(
    train_dataset,
    validation_data=val_dataset,

    initial_epoch=(
        FINETUNE_INITIAL_EPOCH
    ),

    epochs=FINETUNE_FINAL_EPOCH,

    callbacks=finetune_callbacks,
    verbose=1
)


# ------------------------------------------------------------
# Convert fine-tuning history to a table
# ------------------------------------------------------------
finetune_history_df = pd.DataFrame(
    finetune_history.history
)

finetune_history_df.insert(
    0,
    "epoch",
    np.arange(
        FINETUNE_INITIAL_EPOCH + 1,
        FINETUNE_INITIAL_EPOCH + 1 +
        len(finetune_history_df)
    )
)

finetune_history_df.insert(
    1,
    "stage",
    "finetune"
)


# ------------------------------------------------------------
# Save fine-tuning history atomically
# ------------------------------------------------------------
temporary_history_path = (
    FINETUNE_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )
)

finetune_history_df.to_csv(
    temporary_history_path,
    index=False
)

os.replace(
    temporary_history_path,
    FINETUNE_HISTORY_PATH
)


# ------------------------------------------------------------
# Combine frozen and fine-tuning histories
# ------------------------------------------------------------
COMBINED_HISTORY_PATH = (
    METRICS_DIR /
    "combined_training_history.csv"
)

combined_history_df = pd.concat(
    [
        frozen_history_df,
        finetune_history_df
    ],
    ignore_index=True,
    sort=False
)

temporary_combined_path = (
    COMBINED_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )
)

combined_history_df.to_csv(
    temporary_combined_path,
    index=False
)

os.replace(
    temporary_combined_path,
    COMBINED_HISTORY_PATH
)


# ------------------------------------------------------------
# Verify the global best model
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "Global best checkpoint is missing."
)

assert LAST_MODEL_PATH.exists(), (
    "Fine-tuning last checkpoint is missing."
)

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

best_model_predictions = best_model(
    sample_images[:2],
    training=False
).numpy()

assert np.isfinite(
    best_model_predictions
).all(), (
    "Global best model produced NaN or Inf."
)


# ------------------------------------------------------------
# Fine-tuning summary
# ------------------------------------------------------------
best_finetune_epoch_index = int(
    finetune_history_df[
        "val_auc"
    ].idxmax()
)

best_finetune_row = (
    finetune_history_df.loc[
        best_finetune_epoch_index
    ]
)

global_best_history_index = int(
    combined_history_df[
        "val_auc"
    ].idxmax()
)

global_best_history_row = (
    combined_history_df.loc[
        global_best_history_index
    ]
)

finetune_summary = {
    "run_id": RUN_ID,
    "stage": "finetune",
    "epochs_completed": int(
        len(finetune_history_df)
    ),
    "maximum_epochs": FINETUNE_EPOCHS,
    "initial_epoch": (
        FINETUNE_INITIAL_EPOCH
    ),
    "final_epoch_number": int(
        finetune_history_df[
            "epoch"
        ].max()
    ),
    "unfrozen_layer_count": len(
        unfrozen_layer_names
    ),
    "frozen_batch_norm_count": len(
        frozen_batch_norm_names
    ),
    "trainable_parameters": (
        finetune_trainable_parameters
    ),
    "best_finetune_epoch": int(
        best_finetune_row["epoch"]
    ),
    "best_finetune_val_auc": float(
        best_finetune_row["val_auc"]
    ),
    "global_best_epoch": int(
        global_best_history_row["epoch"]
    ),
    "global_best_stage": str(
        global_best_history_row["stage"]
    ),
    "global_best_val_auc": float(
        global_best_history_row["val_auc"]
    ),
    "best_model_path": str(
        BEST_MODEL_PATH
    ),
    "last_model_path": str(
        LAST_MODEL_PATH
    )
}

temporary_summary_path = (
    FINETUNE_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_summary_path,
    "w",
    encoding="utf-8"
) as summary_file:
    json.dump(
        finetune_summary,
        summary_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_summary_path,
    FINETUNE_SUMMARY_PATH
)


# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("\n" + "=" * 90)
print("FINE-TUNING COMPLETED")
print("=" * 90)

print(
    "Fine-tune epochs completed:",
    len(finetune_history_df)
)

print(
    "Best fine-tune epoch     :",
    int(best_finetune_row["epoch"])
)

print(
    "Best fine-tune val AUC   :",
    float(best_finetune_row["val_auc"])
)

print(
    "Global best epoch        :",
    int(global_best_history_row["epoch"])
)

print(
    "Global best stage        :",
    global_best_history_row["stage"]
)

print(
    "Global best val AUC      :",
    float(global_best_history_row["val_auc"])
)

print(
    "Best checkpoint          :",
    BEST_MODEL_PATH
)

print(
    "Combined history         :",
    COMBINED_HISTORY_PATH
)

print(
    "Fine-tune summary        :",
    FINETUNE_SUMMARY_PATH
)

print("=" * 90)
print("PARTIAL FINE-TUNING       : PASS")
print("BATCH NORMALIZATION       : FROZEN")
print("GLOBAL BEST CHECKPOINT    : VERIFIED")
print("COMBINED HISTORY          : SAVED")
print("CELL 13 COMPLETED")
print("=" * 90)



PARTIAL DENSENET121 FINE-TUNING
Best frozen checkpoint loaded: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/best.keras

------------------------------------------------------------------------------------------
FINE-TUNING CONFIGURATION
------------------------------------------------------------------------------------------
Initial checkpoint       : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/best.keras
Fine-tuning learning rate: 1e-05
Maximum fine-tune epochs : 20
Epoch numbering          : 13 to 32
Unfrozen backbone layers : 80
Frozen BatchNorm layers  : 32
Total parameters         : 7,038,529
Trainable parameters     : 2,130,945
Non-trainable parameters : 4,907,584

First five unfrozen layers:
 - conv5_block1_0_relu
 - conv5_block1_1_conv
 - conv5_block1_1_relu
 -

In [21]:
# ============================================================
# CELL 14 — FINAL VALIDATION THRESHOLD AND TEST EVALUATION
# ============================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    matthews_corrcoef
)

print("\n" + "=" * 95)
print("FINAL VALIDATION THRESHOLD AND INDEPENDENT TEST EVALUATION")
print("=" * 95)


# ------------------------------------------------------------
# Load the global best checkpoint
# ------------------------------------------------------------
assert BEST_MODEL_PATH.exists(), (
    "Global best checkpoint is missing."
)

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

print("Global best model:", BEST_MODEL_PATH)
print(
    "Global best validation AUC:",
    finetune_summary["global_best_val_auc"]
)


# ------------------------------------------------------------
# Prediction helper
# ------------------------------------------------------------
def predict_dataset(model_object, dataset):
    labels = []
    probabilities = []

    for image_batch, label_batch in dataset:

        batch_probabilities = (
            model_object(
                image_batch,
                training=False
            )
            .numpy()
            .reshape(-1)
        )

        probabilities.extend(
            batch_probabilities.tolist()
        )

        labels.extend(
            label_batch
            .numpy()
            .astype(int)
            .reshape(-1)
            .tolist()
        )

    labels = np.asarray(
        labels,
        dtype=np.int32
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    assert np.isfinite(
        probabilities
    ).all(), (
        "NaN or Inf detected in predictions."
    )

    assert (
        (probabilities >= 0.0) &
        (probabilities <= 1.0)
    ).all(), (
        "Prediction outside [0, 1]."
    )

    return labels, probabilities


# ------------------------------------------------------------
# Validation predictions
# Validation is used for threshold selection.
# ------------------------------------------------------------
print("\nGenerating validation predictions...")

validation_labels, validation_probabilities = (
    predict_dataset(
        best_model,
        val_dataset
    )
)

assert len(validation_labels) == actual_counts["val"]["total"]


# ------------------------------------------------------------
# Validation-only Youden J threshold selection
# ------------------------------------------------------------
validation_fpr, validation_tpr, validation_thresholds = (
    roc_curve(
        validation_labels,
        validation_probabilities
    )
)

finite_threshold_mask = np.isfinite(
    validation_thresholds
)

finite_fpr = validation_fpr[
    finite_threshold_mask
]

finite_tpr = validation_tpr[
    finite_threshold_mask
]

finite_thresholds = validation_thresholds[
    finite_threshold_mask
]

youden_j_values = (
    finite_tpr -
    finite_fpr
)

best_threshold_index = int(
    np.argmax(
        youden_j_values
    )
)

SELECTED_THRESHOLD = float(
    finite_thresholds[
        best_threshold_index
    ]
)

SELECTED_YOUDEN_J = float(
    youden_j_values[
        best_threshold_index
    ]
)

validation_predictions = (
    validation_probabilities >=
    SELECTED_THRESHOLD
).astype(int)

validation_auc = roc_auc_score(
    validation_labels,
    validation_probabilities
)

validation_f1 = f1_score(
    validation_labels,
    validation_predictions,
    zero_division=0
)

print("\nValidation-only threshold selection:")
print("Selected threshold :", SELECTED_THRESHOLD)
print("Youden J           :", SELECTED_YOUDEN_J)
print("Validation ROC-AUC :", validation_auc)
print("Validation F1      :", validation_f1)


# ------------------------------------------------------------
# Independent test predictions
# Test is evaluated only after threshold selection.
# ------------------------------------------------------------
print("\nGenerating independent test predictions...")

test_labels, test_probabilities = (
    predict_dataset(
        best_model,
        test_dataset
    )
)

assert len(test_labels) == actual_counts["test"]["total"]

test_predictions = (
    test_probabilities >=
    SELECTED_THRESHOLD
).astype(int)

default_test_predictions = (
    test_probabilities >= 0.5
).astype(int)


# ------------------------------------------------------------
# Metric calculation function
# ------------------------------------------------------------
def calculate_binary_metrics(
    labels,
    probabilities,
    predictions,
    threshold
):
    matrix = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    )

    true_negative = int(matrix[0, 0])
    false_positive = int(matrix[0, 1])
    false_negative = int(matrix[1, 0])
    true_positive = int(matrix[1, 1])

    specificity_denominator = (
        true_negative +
        false_positive
    )

    specificity = (
        true_negative /
        specificity_denominator
        if specificity_denominator > 0
        else 0.0
    )

    return {
        "threshold": float(threshold),
        "sample_count": int(len(labels)),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions
            )
        ),
        "precision": float(
            precision_score(
                labels,
                predictions,
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                labels,
                predictions,
                zero_division=0
            )
        ),
        "specificity": float(
            specificity
        ),
        "f1_score": float(
            f1_score(
                labels,
                predictions,
                zero_division=0
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                probabilities
            )
        ),
        "pr_auc": float(
            average_precision_score(
                labels,
                probabilities
            )
        ),
        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                labels,
                predictions
            )
        ),
        "true_negative": true_negative,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_positive": true_positive
    }


# ------------------------------------------------------------
# Image-level metrics
# ------------------------------------------------------------
image_level_metrics = (
    calculate_binary_metrics(
        labels=test_labels,
        probabilities=test_probabilities,
        predictions=test_predictions,
        threshold=SELECTED_THRESHOLD
    )
)

default_threshold_metrics = (
    calculate_binary_metrics(
        labels=test_labels,
        probabilities=test_probabilities,
        predictions=default_test_predictions,
        threshold=0.5
    )
)


# ------------------------------------------------------------
# Align test predictions with the standard manifest
# ------------------------------------------------------------
test_manifest = (
    model_manifest.loc[
        model_manifest["split"] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

expected_test_paths = [
    str(path)
    for path in dataset_records[
        "test"
    ]["paths"]
]

assert test_manifest[
    "output_path"
].tolist() == expected_test_paths, (
    "Test prediction order does not match "
    "the standard manifest."
)

assert np.array_equal(
    test_manifest["label"]
    .map(LABEL_MAP)
    .to_numpy(dtype=int),
    test_labels
), (
    "Test labels do not match manifest labels."
)

test_predictions_df = test_manifest[
    [
        "sample_id",
        "source_video",
        "frame_index",
        "face_index",
        "label",
        "sha256",
        "output_path"
    ]
].copy()

test_predictions_df[
    "true_label"
] = test_labels

test_predictions_df[
    "fake_probability"
] = test_probabilities

test_predictions_df[
    "selected_threshold"
] = SELECTED_THRESHOLD

test_predictions_df[
    "predicted_label"
] = test_predictions

test_predictions_df[
    "predicted_class"
] = np.where(
    test_predictions == 1,
    "fake",
    "real"
)

test_predictions_df[
    "is_correct"
] = (
    test_predictions_df[
        "true_label"
    ] ==
    test_predictions_df[
        "predicted_label"
    ]
)


# ------------------------------------------------------------
# Source/video-level aggregation
# Multiple face ROIs belonging to the same source are averaged.
# ------------------------------------------------------------
label_consistency = (
    test_predictions_df
    .groupby("source_video")[
        "true_label"
    ]
    .nunique()
)

assert (
    label_consistency == 1
).all(), (
    "A source_video has conflicting labels."
)

source_predictions_df = (
    test_predictions_df
    .groupby(
        "source_video",
        as_index=False
    )
    .agg(
        true_label=(
            "true_label",
            "first"
        ),
        fake_probability=(
            "fake_probability",
            "mean"
        ),
        roi_count=(
            "sample_id",
            "count"
        )
    )
)

source_predictions_df[
    "selected_threshold"
] = SELECTED_THRESHOLD

source_predictions_df[
    "predicted_label"
] = (
    source_predictions_df[
        "fake_probability"
    ] >= SELECTED_THRESHOLD
).astype(int)

source_predictions_df[
    "predicted_class"
] = np.where(
    source_predictions_df[
        "predicted_label"
    ] == 1,
    "fake",
    "real"
)

source_predictions_df[
    "is_correct"
] = (
    source_predictions_df[
        "true_label"
    ] ==
    source_predictions_df[
        "predicted_label"
    ]
)

source_level_metrics = (
    calculate_binary_metrics(
        labels=source_predictions_df[
            "true_label"
        ].to_numpy(dtype=int),

        probabilities=source_predictions_df[
            "fake_probability"
        ].to_numpy(dtype=float),

        predictions=source_predictions_df[
            "predicted_label"
        ].to_numpy(dtype=int),

        threshold=SELECTED_THRESHOLD
    )
)


# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------
classification_report_dict = (
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1],
        target_names=[
            "Real",
            "Fake"
        ],
        output_dict=True,
        zero_division=0
    )
)

classification_report_df = (
    pd.DataFrame(
        classification_report_dict
    )
    .transpose()
)


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------
IMAGE_METRICS_PATH = (
    METRICS_DIR /
    "final_image_level_metrics.csv"
)

SOURCE_METRICS_PATH = (
    METRICS_DIR /
    "final_source_level_metrics.csv"
)

DEFAULT_METRICS_PATH = (
    METRICS_DIR /
    "default_threshold_metrics.csv"
)

CLASSIFICATION_REPORT_PATH = (
    METRICS_DIR /
    "classification_report.csv"
)

THRESHOLD_PATH = (
    METRICS_DIR /
    "validation_threshold.json"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR /
    "test_image_predictions.csv"
)

SOURCE_PREDICTIONS_PATH = (
    PREDICTION_DIR /
    "test_source_predictions.csv"
)


# ------------------------------------------------------------
# Atomic table writer
# ------------------------------------------------------------
def atomic_write_csv(
    dataframe,
    output_path
):
    output_path = Path(output_path)

    temporary_path = (
        output_path.with_suffix(
            ".csv.tmp"
        )
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        output_path
    )


atomic_write_csv(
    pd.DataFrame(
        [image_level_metrics]
    ),
    IMAGE_METRICS_PATH
)

atomic_write_csv(
    pd.DataFrame(
        [source_level_metrics]
    ),
    SOURCE_METRICS_PATH
)

atomic_write_csv(
    pd.DataFrame(
        [default_threshold_metrics]
    ),
    DEFAULT_METRICS_PATH
)

atomic_write_csv(
    classification_report_df.reset_index(
        names="class"
    ),
    CLASSIFICATION_REPORT_PATH
)

atomic_write_csv(
    test_predictions_df,
    TEST_PREDICTIONS_PATH
)

atomic_write_csv(
    source_predictions_df,
    SOURCE_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# Save validation threshold atomically
# ------------------------------------------------------------
threshold_data = {
    "run_id": RUN_ID,
    "selection_dataset": "validation",
    "selection_method": "Youden J",
    "selected_threshold": SELECTED_THRESHOLD,
    "youden_j": SELECTED_YOUDEN_J,
    "validation_roc_auc": float(
        validation_auc
    ),
    "validation_f1_at_selected_threshold": float(
        validation_f1
    ),
    "test_set_used_for_threshold_selection": False
}

temporary_threshold_path = (
    THRESHOLD_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_threshold_path,
    "w",
    encoding="utf-8"
) as threshold_file:
    json.dump(
        threshold_data,
        threshold_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_threshold_path,
    THRESHOLD_PATH
)


# ------------------------------------------------------------
# Plot configuration
# All report figures use English labels.
# ------------------------------------------------------------
sns.set_theme(
    style="whitegrid",
    context="notebook"
)

TRAIN_COLOR = "#2878B5"
VALIDATION_COLOR = "#E07A2D"
REAL_COLOR = "#2A9D8F"
FAKE_COLOR = "#D1495B"


def save_figure_with_quality_gate(
    figure,
    output_path
):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.stem}.tmp.png"
    )

    figure.savefig(
        temporary_path,
        dpi=150,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(figure)

    with Image.open(
        temporary_path
    ) as image_object:
        image_size = image_object.size

    assert min(image_size) >= 600, (
        f"Figure resolution is too low: "
        f"{image_size}"
    )

    os.replace(
        temporary_path,
        output_path
    )

    return image_size


# ------------------------------------------------------------
# Figure 1 — Combined training curves
# ------------------------------------------------------------
TRAINING_CURVES_PATH = (
    FIGURE_DIR /
    "training_validation_curves.png"
)

figure, axes = plt.subplots(
    3,
    1,
    figsize=(11, 14),
    dpi=150,
    sharex=True
)

epoch_values = combined_history_df[
    "epoch"
].to_numpy()

axes[0].plot(
    epoch_values,
    combined_history_df["loss"],
    label="Training Loss",
    color=TRAIN_COLOR,
    linewidth=2
)

axes[0].plot(
    epoch_values,
    combined_history_df["val_loss"],
    label="Validation Loss",
    color=VALIDATION_COLOR,
    linewidth=2,
    linestyle="--"
)

axes[0].set_title(
    "Training and Validation Loss",
    fontsize=14,
    fontweight="bold"
)

axes[0].set_ylabel(
    "Binary Cross-Entropy Loss",
    fontsize=11
)

axes[1].plot(
    epoch_values,
    combined_history_df["auc"],
    label="Training ROC-AUC",
    color=TRAIN_COLOR,
    linewidth=2
)

axes[1].plot(
    epoch_values,
    combined_history_df["val_auc"],
    label="Validation ROC-AUC",
    color=VALIDATION_COLOR,
    linewidth=2,
    linestyle="--"
)

axes[1].set_title(
    "Training and Validation ROC-AUC",
    fontsize=14,
    fontweight="bold"
)

axes[1].set_ylabel(
    "ROC-AUC",
    fontsize=11
)

axes[2].plot(
    epoch_values,
    combined_history_df["accuracy"],
    label="Training Accuracy",
    color=TRAIN_COLOR,
    linewidth=2
)

axes[2].plot(
    epoch_values,
    combined_history_df["val_accuracy"],
    label="Validation Accuracy",
    color=VALIDATION_COLOR,
    linewidth=2,
    linestyle="--"
)

axes[2].set_title(
    "Training and Validation Accuracy",
    fontsize=14,
    fontweight="bold"
)

axes[2].set_xlabel(
    "Epoch",
    fontsize=11
)

axes[2].set_ylabel(
    "Accuracy",
    fontsize=11
)

for axis in axes:
    axis.axvline(
        FROZEN_EPOCHS_COMPLETED + 0.5,
        color="#6C757D",
        linestyle=":",
        linewidth=2,
        label="Fine-Tuning Start"
    )

    axis.legend(
        frameon=True,
        fontsize=10
    )

    axis.grid(
        True,
        alpha=0.25
    )

figure.tight_layout()

training_curve_size = (
    save_figure_with_quality_gate(
        figure,
        TRAINING_CURVES_PATH
    )
)


# ------------------------------------------------------------
# Figure 2 — Image and source confusion matrices
# ------------------------------------------------------------
CONFUSION_MATRIX_PATH = (
    FIGURE_DIR /
    "test_confusion_matrices.png"
)

image_confusion = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1]
)

source_confusion = confusion_matrix(
    source_predictions_df["true_label"],
    source_predictions_df["predicted_label"],
    labels=[0, 1]
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.5),
    dpi=150
)

sns.heatmap(
    image_confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    square=True,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=axes[0]
)

axes[0].set_title(
    "Image-Level Test Confusion Matrix",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel(
    "Predicted Label",
    fontsize=11
)

axes[0].set_ylabel(
    "True Label",
    fontsize=11
)

sns.heatmap(
    source_confusion,
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False,
    square=True,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=axes[1]
)

axes[1].set_title(
    "Source-Level Test Confusion Matrix",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel(
    "Predicted Label",
    fontsize=11
)

axes[1].set_ylabel(
    "True Label",
    fontsize=11
)

figure.tight_layout()

confusion_matrix_size = (
    save_figure_with_quality_gate(
        figure,
        CONFUSION_MATRIX_PATH
    )
)


# ------------------------------------------------------------
# Figure 3 — ROC and Precision-Recall curves
# ------------------------------------------------------------
ROC_PR_PATH = (
    FIGURE_DIR /
    "test_roc_pr_curves.png"
)

test_fpr, test_tpr, _ = roc_curve(
    test_labels,
    test_probabilities
)

test_precision_curve, test_recall_curve, _ = (
    precision_recall_curve(
        test_labels,
        test_probabilities
    )
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5),
    dpi=150
)

axes[0].plot(
    test_fpr,
    test_tpr,
    color=TRAIN_COLOR,
    linewidth=2.5,
    label=(
        f"ROC-AUC = "
        f"{image_level_metrics['roc_auc']:.3f}"
    )
)

axes[0].plot(
    [0, 1],
    [0, 1],
    color="#777777",
    linestyle="--",
    linewidth=1.5,
    label="Random Classifier"
)

axes[0].set_title(
    "Image-Level Test ROC Curve",
    fontsize=13,
    fontweight="bold"
)

axes[0].set_xlabel(
    "False Positive Rate",
    fontsize=11
)

axes[0].set_ylabel(
    "True Positive Rate",
    fontsize=11
)

axes[0].legend(
    frameon=True
)

axes[0].grid(
    True,
    alpha=0.25
)

axes[1].plot(
    test_recall_curve,
    test_precision_curve,
    color=FAKE_COLOR,
    linewidth=2.5,
    label=(
        f"PR-AUC = "
        f"{image_level_metrics['pr_auc']:.3f}"
    )
)

axes[1].set_title(
    "Image-Level Test Precision-Recall Curve",
    fontsize=13,
    fontweight="bold"
)

axes[1].set_xlabel(
    "Recall",
    fontsize=11
)

axes[1].set_ylabel(
    "Precision",
    fontsize=11
)

axes[1].legend(
    frameon=True
)

axes[1].grid(
    True,
    alpha=0.25
)

figure.tight_layout()

roc_pr_size = (
    save_figure_with_quality_gate(
        figure,
        ROC_PR_PATH
    )
)


# ------------------------------------------------------------
# Figure 4 — Probability distribution
# ------------------------------------------------------------
PROBABILITY_DISTRIBUTION_PATH = (
    FIGURE_DIR /
    "test_probability_distribution.png"
)

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150
)

sns.histplot(
    test_probabilities[
        test_labels == 0
    ],
    bins=20,
    stat="density",
    alpha=0.55,
    color=REAL_COLOR,
    label="Real",
    ax=axis
)

sns.histplot(
    test_probabilities[
        test_labels == 1
    ],
    bins=20,
    stat="density",
    alpha=0.55,
    color=FAKE_COLOR,
    label="Fake",
    ax=axis
)

axis.axvline(
    SELECTED_THRESHOLD,
    color="#222222",
    linestyle="--",
    linewidth=2,
    label=(
        f"Validation Threshold = "
        f"{SELECTED_THRESHOLD:.3f}"
    )
)

axis.set_title(
    "Test Fake-Probability Distribution",
    fontsize=14,
    fontweight="bold"
)

axis.set_xlabel(
    "Predicted Fake Probability",
    fontsize=11
)

axis.set_ylabel(
    "Density",
    fontsize=11
)

axis.legend(
    frameon=True
)

axis.grid(
    True,
    alpha=0.25
)

figure.tight_layout()

probability_size = (
    save_figure_with_quality_gate(
        figure,
        PROBABILITY_DISTRIBUTION_PATH
    )
)


# ------------------------------------------------------------
# Final evaluation summary
# ------------------------------------------------------------
FINAL_EVALUATION_PATH = (
    METRICS_DIR /
    "final_evaluation_summary.json"
)

final_evaluation_summary = {
    "run_id": RUN_ID,
    "model": "DenseNet121",
    "global_best_epoch": int(
        finetune_summary[
            "global_best_epoch"
        ]
    ),
    "global_best_stage": str(
        finetune_summary[
            "global_best_stage"
        ]
    ),
    "global_best_validation_auc": float(
        finetune_summary[
            "global_best_val_auc"
        ]
    ),
    "threshold_selection": threshold_data,
    "image_level_test_metrics": (
        image_level_metrics
    ),
    "source_level_test_metrics": (
        source_level_metrics
    ),
    "default_threshold_test_metrics": (
        default_threshold_metrics
    ),
    "test_used_once_for_final_evaluation": True,
    "figures": {
        "training_curves": {
            "path": str(
                TRAINING_CURVES_PATH
            ),
            "size": training_curve_size
        },
        "confusion_matrices": {
            "path": str(
                CONFUSION_MATRIX_PATH
            ),
            "size": confusion_matrix_size
        },
        "roc_pr_curves": {
            "path": str(
                ROC_PR_PATH
            ),
            "size": roc_pr_size
        },
        "probability_distribution": {
            "path": str(
                PROBABILITY_DISTRIBUTION_PATH
            ),
            "size": probability_size
        }
    }
}

temporary_final_path = (
    FINAL_EVALUATION_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_final_path,
    "w",
    encoding="utf-8"
) as final_file:
    json.dump(
        final_evaluation_summary,
        final_file,
        indent=4,
        ensure_ascii=False
    )

os.replace(
    temporary_final_path,
    FINAL_EVALUATION_PATH
)


# ------------------------------------------------------------
# Print final test results
# ------------------------------------------------------------
print("\n" + "=" * 95)
print("FINAL INDEPENDENT TEST RESULTS")
print("=" * 95)

print(
    "Validation-selected threshold:",
    f"{SELECTED_THRESHOLD:.6f}"
)

print("\nIMAGE-LEVEL TEST METRICS")

for metric_name, metric_value in (
    image_level_metrics.items()
):
    if isinstance(metric_value, float):
        print(
            f"{metric_name:35}: "
            f"{metric_value:.6f}"
        )
    else:
        print(
            f"{metric_name:35}: "
            f"{metric_value}"
        )

print("\nSOURCE-LEVEL TEST METRICS")

for metric_name, metric_value in (
    source_level_metrics.items()
):
    if isinstance(metric_value, float):
        print(
            f"{metric_name:35}: "
            f"{metric_value:.6f}"
        )
    else:
        print(
            f"{metric_name:35}: "
            f"{metric_value}"
        )

print("\nSaved figures:")
print(" -", TRAINING_CURVES_PATH)
print(" -", CONFUSION_MATRIX_PATH)
print(" -", ROC_PR_PATH)
print(" -", PROBABILITY_DISTRIBUTION_PATH)

print("=" * 95)
print("VALIDATION THRESHOLD SELECTION : PASS")
print("INDEPENDENT TEST EVALUATION    : PASS")
print("IMAGE-LEVEL METRICS            : SAVED")
print("SOURCE-LEVEL METRICS           : SAVED")
print("FIGURE QUALITY GATES           : PASS")
print("CELL 14 COMPLETED")
print("=" * 95)



FINAL VALIDATION THRESHOLD AND INDEPENDENT TEST EVALUATION
Global best model: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/checkpoints/best.keras
Global best validation AUC: 0.7387096881866455

Generating validation predictions...

Validation-only threshold selection:
Selected threshold : 0.4685651659965515
Youden J           : 0.3588194921070693
Validation ROC-AUC : 0.7383207504003659
Validation F1      : 0.7069486404833837

Generating independent test predictions...

FINAL INDEPENDENT TEST RESULTS
Validation-selected threshold: 0.468565

IMAGE-LEVEL TEST METRICS
threshold                          : 0.468565
sample_count                       : 302
accuracy                           : 0.602649
balanced_accuracy                  : 0.596944
precision                          : 0.588235
recall                             : 0.769231
specificity                        : 0.424658
f1_score      

In [22]:
# ============================================================
# RUN SUMMARY
# ============================================================
print("RUN_ID   :", RUN_ID)
print("RUN_ROOT :", RUN_ROOT)
print("MANIFEST :", MANIFEST_PATH)


RUN_ID   : 20260808_1214_eye_densenet121_seed42
RUN_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42
MANIFEST : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/metadata/model_manifest.csv


In [23]:
# ============================================================
# OPTIONAL RECOVERY CELL — RESTORE LATEST / SPECIFIC EYE RUN
# No training is performed here.
# ============================================================

from pathlib import Path
import json
import pandas as pd
import tensorflow as tf

MODEL_RESULTS_ROOT = RESULTS_ROOT / "DenseNet121_Eye_Results"

requested_run_id = CONFIG.get("resume_run_id")

if requested_run_id:
    RUN_ID = str(requested_run_id)
    RUN_ROOT = MODEL_RESULTS_ROOT / RUN_ID
else:
    candidates = sorted(
        [p for p in MODEL_RESULTS_ROOT.glob("*") if p.is_dir()],
        key=lambda p: p.name,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No completed eye run folders found under: {MODEL_RESULTS_ROOT}"
        )
    RUN_ROOT = candidates[-1]
    RUN_ID = RUN_ROOT.name

CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
LOG_DIR = RUN_ROOT / "logs"
METRICS_DIR = RUN_ROOT / "metrics"
PREDICTION_DIR = RUN_ROOT / "predictions"
FIGURE_DIR = RUN_ROOT / "figures"
METADATA_DIR = RUN_ROOT / "metadata"

BEST_MODEL_PATH = CHECKPOINT_DIR / "best.keras"
LAST_MODEL_PATH = CHECKPOINT_DIR / "last.keras"
MANIFEST_PATH = METADATA_DIR / "model_manifest.csv"
LEAKAGE_REPORT_PATH = METADATA_DIR / "leakage_check.csv"

required = [
    BEST_MODEL_PATH,
    LAST_MODEL_PATH,
    MANIFEST_PATH,
    LEAKAGE_REPORT_PATH,
]

for path in required:
    assert path.exists() and path.stat().st_size > 0, (
        f"Required recovery file missing or empty: {path}"
    )

model_manifest = pd.read_csv(MANIFEST_PATH)
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

print("\nRECOVERY PASS")
print("RUN_ID   :", RUN_ID)
print("RUN_ROOT :", RUN_ROOT)
print("Rows     :", len(model_manifest))



RECOVERY PASS
RUN_ID   : 20260808_1214_eye_densenet121_seed42
RUN_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42
Rows     : 2986


In [24]:
# ============================================================
# CELL 15 — FINAL AUDIT AND ZIP EXPORT
# Run this after the Recovery Cell.
# ============================================================

import os
import json
import shutil
import hashlib
import zipfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn
import PIL
import yaml
import tensorflow as tf

from PIL import Image


print("\n" + "=" * 100)
print("FINAL AUDIT, CLEAN-LOAD INFERENCE TEST AND ZIP EXPORT")
print("=" * 100)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def atomic_write_text(output_path, content):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.name}.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8"
    ) as output_file:
        output_file.write(content)

    os.replace(
        temporary_path,
        output_path
    )


def atomic_write_json(output_path, content):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.name}.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8"
    ) as output_file:
        json.dump(
            content,
            output_file,
            indent=4,
            ensure_ascii=False
        )

    os.replace(
        temporary_path,
        output_path
    )


def atomic_write_csv(dataframe, output_path):
    output_path = Path(output_path)

    temporary_path = (
        output_path.parent /
        f"{output_path.name}.tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        output_path
    )


def calculate_sha256(file_path):
    sha256_object = hashlib.sha256()

    with open(file_path, "rb") as file_stream:

        while True:
            file_chunk = file_stream.read(
                1024 * 1024
            )

            if not file_chunk:
                break

            sha256_object.update(
                file_chunk
            )

    return sha256_object.hexdigest()


# ------------------------------------------------------------
# Required directories
# ------------------------------------------------------------
required_directories = {
    "checkpoints": CHECKPOINT_DIR,
    "logs": LOG_DIR,
    "metrics": METRICS_DIR,
    "predictions": PREDICTION_DIR,
    "figures": FIGURE_DIR,
    "metadata": METADATA_DIR
}

print("\nRequired directories:")
print("-" * 100)

for directory_name, directory_path in (
    required_directories.items()
):
    assert directory_path.exists(), (
        f"Missing directory: {directory_path}"
    )

    print(
        f"{directory_name:15}: FOUND"
    )


# ------------------------------------------------------------
# Required final files
# ------------------------------------------------------------
required_files = {
    "resolved_config": (
        RUN_ROOT /
        "config_resolved.yaml"
    ),

    "best_model": BEST_MODEL_PATH,

    "last_model": LAST_MODEL_PATH,

    "model_manifest": MANIFEST_PATH,

    "leakage_report": LEAKAGE_REPORT_PATH,

    "accounting_report": (
        METADATA_DIR /
        "accounting_summary.json"
    ),

    "smoke_test_report": (
        METRICS_DIR /
        "smoke_test_report.json"
    ),

    "frozen_history": (
        METRICS_DIR /
        "frozen_training_history.csv"
    ),

    "finetune_history": (
        METRICS_DIR /
        "finetune_training_history.csv"
    ),

    "combined_history": (
        METRICS_DIR /
        "combined_training_history.csv"
    ),

    "image_metrics": (
        METRICS_DIR /
        "final_image_level_metrics.csv"
    ),

    "source_metrics": (
        METRICS_DIR /
        "final_source_level_metrics.csv"
    ),

    "classification_report": (
        METRICS_DIR /
        "classification_report.csv"
    ),

    "validation_threshold": (
        METRICS_DIR /
        "validation_threshold.json"
    ),

    "final_evaluation": (
        METRICS_DIR /
        "final_evaluation_summary.json"
    ),

    "image_predictions": (
        PREDICTION_DIR /
        "test_image_predictions.csv"
    ),

    "source_predictions": (
        PREDICTION_DIR /
        "test_source_predictions.csv"
    ),

    "training_curves": (
        FIGURE_DIR /
        "training_validation_curves.png"
    ),

    "confusion_matrices": (
        FIGURE_DIR /
        "test_confusion_matrices.png"
    ),

    "roc_pr_curves": (
        FIGURE_DIR /
        "test_roc_pr_curves.png"
    ),

    "probability_distribution": (
        FIGURE_DIR /
        "test_probability_distribution.png"
    )
}


print("\nRequired files:")
print("-" * 100)

for file_name, file_path in required_files.items():

    assert file_path.exists(), (
        f"Missing file: {file_path}"
    )

    assert file_path.stat().st_size > 0, (
        f"Empty file: {file_path}"
    )

    print(
        f"{file_name:28}: FOUND"
    )


# ------------------------------------------------------------
# Reload final tables
# ------------------------------------------------------------
saved_manifest = pd.read_csv(
    MANIFEST_PATH
)

saved_leakage_report = pd.read_csv(
    LEAKAGE_REPORT_PATH
)

saved_image_metrics = pd.read_csv(
    required_files["image_metrics"]
)

saved_source_metrics = pd.read_csv(
    required_files["source_metrics"]
)

saved_image_predictions = pd.read_csv(
    required_files["image_predictions"]
)

saved_source_predictions = pd.read_csv(
    required_files["source_predictions"]
)

saved_combined_history = pd.read_csv(
    required_files["combined_history"]
)


# ------------------------------------------------------------
# Table quality gates
# ------------------------------------------------------------
assert len(saved_manifest) == len(model_manifest)
assert saved_manifest["sample_id"].is_unique
assert saved_manifest["sha256"].notna().all()
assert saved_manifest["output_path"].notna().all()

assert len(saved_image_predictions) == int((saved_manifest["split"] == "test").sum())
assert len(saved_source_predictions) == saved_image_predictions["source_video"].nunique()
assert len(saved_combined_history) > 0

assert (
    saved_leakage_report[
        "source_video_overlap"
    ] == 0
).all()

assert (
    saved_leakage_report[
        "sha256_overlap"
    ] == 0
).all()

assert np.isfinite(
    saved_image_metrics.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

assert np.isfinite(
    saved_source_metrics.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

print("\nTable quality gates: PASS")


# ------------------------------------------------------------
# Clean-load inference test
# ------------------------------------------------------------
print("\nRunning clean-load inference test...")

fresh_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

fresh_images, fresh_labels = next(
    iter(test_dataset)
)

fresh_probabilities = (
    fresh_model(
        fresh_images[:4],
        training=False
    )
    .numpy()
    .reshape(-1)
)

assert fresh_probabilities.shape == (4,)

assert np.isfinite(
    fresh_probabilities
).all()

assert (
    (fresh_probabilities >= 0.0) &
    (fresh_probabilities <= 1.0)
).all()

print(
    "Clean-load probabilities:",
    fresh_probabilities.tolist()
)

print("Clean-load inference: PASS")


# ------------------------------------------------------------
# Figure quality gates
# ------------------------------------------------------------
figure_quality_records = []

figure_keys = [
    "training_curves",
    "confusion_matrices",
    "roc_pr_curves",
    "probability_distribution"
]

print("\nFigure quality checks:")
print("-" * 100)

for figure_name in figure_keys:

    figure_path = required_files[
        figure_name
    ]

    with Image.open(
        figure_path
    ) as figure_image:
        width, height = figure_image.size

    minimum_side = min(
        width,
        height
    )

    assert minimum_side >= 600, (
        f"Low-resolution figure: "
        f"{figure_path} = "
        f"{width} x {height}"
    )

    figure_quality_records.append(
        {
            "figure": figure_name,
            "path": str(figure_path),
            "width_px": int(width),
            "height_px": int(height),
            "minimum_side_px": int(
                minimum_side
            ),
            "status": "PASS"
        }
    )

    print(
        f"{figure_name:28}: "
        f"{width} x {height} px | PASS"
    )


FIGURE_AUDIT_PATH = (
    METRICS_DIR /
    "figure_quality_audit.csv"
)

atomic_write_csv(
    pd.DataFrame(
        figure_quality_records
    ),
    FIGURE_AUDIT_PATH
)


# ------------------------------------------------------------
# Runtime dependency snapshot
# ------------------------------------------------------------
REQUIREMENTS_PATH = (
    RUN_ROOT /
    "requirements_snapshot.txt"
)

requirements_text = "\n".join(
    [
        f"python=={platform.python_version()}",
        f"tensorflow=={tf.__version__}",
        f"numpy=={np.__version__}",
        f"pandas=={pd.__version__}",
        f"scikit-learn=={sklearn.__version__}",
        f"matplotlib=={matplotlib.__version__}",
        f"seaborn=={seaborn.__version__}",
        f"pillow=={PIL.__version__}",
        f"pyyaml=={yaml.__version__}"
    ]
) + "\n"

atomic_write_text(
    REQUIREMENTS_PATH,
    requirements_text
)


# ------------------------------------------------------------
# Final metric rows
# ------------------------------------------------------------
image_metrics_row = (
    saved_image_metrics.iloc[0]
)

source_metrics_row = (
    saved_source_metrics.iloc[0]
)


# ------------------------------------------------------------
# Human-readable summary
# ------------------------------------------------------------
RUN_SUMMARY_PATH = (
    RUN_ROOT /
    "RUN_SUMMARY.txt"
)

run_summary_text = f"""
DENSENET121 EYE ROI DEEPFAKE EXPERIMENT
=========================================

Run ID
------
{RUN_ID}

Dataset
-------
Training images   : {int((saved_manifest["split"] == "train").sum())}
Validation images : {int((saved_manifest["split"] == "val").sum())}
Test images       : {int((saved_manifest["split"] == "test").sum())}
Total ROI images  : {len(saved_manifest)}

Model
-----
Architecture      : ImageNet-pretrained DenseNet121
Input             : 224 x 224 RGB eye ROI
Frozen stage      : {int(CONFIG["frozen_epochs"])} configured epochs
Fine-tuning stage : {int(CONFIG["finetune_epochs"])} configured epochs
Global best epoch : {finetune_summary["global_best_epoch"]}
Best val ROC-AUC  : {finetune_summary["global_best_val_auc"]:.6f}

Threshold Selection
-------------------
Dataset           : Validation
Method            : Youden J
Threshold         : {SELECTED_THRESHOLD:.6f}
Test used         : No

Image-Level Independent Test Results
------------------------------------
Accuracy          : {float(image_metrics_row["accuracy"]):.6f}
Balanced Accuracy : {float(image_metrics_row["balanced_accuracy"]):.6f}
Precision         : {float(image_metrics_row["precision"]):.6f}
Recall            : {float(image_metrics_row["recall"]):.6f}
Specificity       : {float(image_metrics_row["specificity"]):.6f}
F1-score          : {float(image_metrics_row["f1_score"]):.6f}
ROC-AUC           : {float(image_metrics_row["roc_auc"]):.6f}
PR-AUC            : {float(image_metrics_row["pr_auc"]):.6f}

Source-Level Independent Test Results
-------------------------------------
Source count      : {int(source_metrics_row["sample_count"])}
Accuracy          : {float(source_metrics_row["accuracy"]):.6f}
Balanced Accuracy : {float(source_metrics_row["balanced_accuracy"]):.6f}
Precision         : {float(source_metrics_row["precision"]):.6f}
Recall            : {float(source_metrics_row["recall"]):.6f}
Specificity       : {float(source_metrics_row["specificity"]):.6f}
F1-score          : {float(source_metrics_row["f1_score"]):.6f}
ROC-AUC           : {float(source_metrics_row["roc_auc"]):.6f}
PR-AUC            : {float(source_metrics_row["pr_auc"]):.6f}

Quality Gates
-------------
Metadata accounting       : PASS
Image-metadata matching   : PASS
Source-video leakage      : PASS
SHA-256 leakage           : PASS
Two-batch training smoke  : PASS
Gradient NaN/Inf          : PASS
Atomic checkpoint         : PASS
Checkpoint reload         : PASS
Clean-load inference      : PASS
Figure resolution         : PASS

Label Mapping
-------------
Real = 0
Fake = 1
""".strip() + "\n"

atomic_write_text(
    RUN_SUMMARY_PATH,
    run_summary_text
)


# ------------------------------------------------------------
# Machine-readable final audit
# ------------------------------------------------------------
FINAL_AUDIT_PATH = (
    RUN_ROOT /
    "final_audit.json"
)

image_metrics_dictionary = {
    column_name: float(
        image_metrics_row[column_name]
    )
    for column_name in (
        saved_image_metrics.columns
    )
}

source_metrics_dictionary = {
    column_name: float(
        source_metrics_row[column_name]
    )
    for column_name in (
        saved_source_metrics.columns
    )
}

final_audit = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "model": "DenseNet121",
    "best_model_path": str(
        BEST_MODEL_PATH
    ),
    "best_model_exists": True,
    "last_model_exists": True,
    "manifest_rows": int(
        len(saved_manifest)
    ),
    "test_prediction_rows": int(
        len(saved_image_predictions)
    ),
    "source_prediction_rows": int(
        len(saved_source_predictions)
    ),
    "combined_training_epochs": int(
        len(saved_combined_history)
    ),
    "source_video_leakage_pass": True,
    "sha256_leakage_pass": True,
    "clean_load_inference_pass": True,
    "figure_quality_pass": True,
    "test_used_only_for_final_evaluation": True,
    "image_level_metrics": (
        image_metrics_dictionary
    ),
    "source_level_metrics": (
        source_metrics_dictionary
    )
}

atomic_write_json(
    FINAL_AUDIT_PATH,
    final_audit
)


# ------------------------------------------------------------
# Artifact checksum manifest
# ------------------------------------------------------------
ARTIFACT_MANIFEST_PATH = (
    RUN_ROOT /
    "artifact_manifest.csv"
)

artifact_files = sorted(
    [
        file_path
        for file_path in RUN_ROOT.rglob("*")
        if (
            file_path.is_file()
            and file_path !=
            ARTIFACT_MANIFEST_PATH
        )
    ],
    key=lambda file_path: str(
        file_path.relative_to(
            RUN_ROOT
        )
    )
)

artifact_records = []

print("\nCalculating artifact checksums...")

for artifact_number, artifact_path in enumerate(
    artifact_files,
    start=1
):
    artifact_records.append(
        {
            "relative_path": str(
                artifact_path.relative_to(
                    RUN_ROOT
                )
            ),
            "size_bytes": int(
                artifact_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                artifact_path
            )
        }
    )

    if (
        artifact_number % 10 == 0
        or artifact_number ==
        len(artifact_files)
    ):
        print(
            f"Artifacts hashed: "
            f"{artifact_number} / "
            f"{len(artifact_files)}"
        )

atomic_write_csv(
    pd.DataFrame(
        artifact_records
    ),
    ARTIFACT_MANIFEST_PATH
)


# ------------------------------------------------------------
# ZIP paths
# ------------------------------------------------------------
ZIP_PATH = (
    RUN_ROOT.parent /
    f"{RUN_ID}.zip"
)

TEMP_ZIP_BASE = (
    RUN_ROOT.parent /
    f"{RUN_ID}.tmp"
)

TEMP_ZIP_PATH = Path(
    str(TEMP_ZIP_BASE) +
    ".zip"
)

ZIP_SHA256_PATH = (
    RUN_ROOT.parent /
    f"{RUN_ID}.zip.sha256"
)

if TEMP_ZIP_PATH.exists():
    TEMP_ZIP_PATH.unlink()


# ------------------------------------------------------------
# Create ZIP
# ------------------------------------------------------------
print("\nCreating final ZIP archive...")

created_zip_path = shutil.make_archive(
    base_name=str(
        TEMP_ZIP_BASE
    ),
    format="zip",
    root_dir=str(
        RUN_ROOT.parent
    ),
    base_dir=RUN_ROOT.name
)

created_zip_path = Path(
    created_zip_path
)

assert created_zip_path == TEMP_ZIP_PATH

assert TEMP_ZIP_PATH.exists()


# ------------------------------------------------------------
# ZIP integrity verification
# ------------------------------------------------------------
with zipfile.ZipFile(
    TEMP_ZIP_PATH,
    mode="r"
) as zip_file:

    corrupt_member = zip_file.testzip()

    assert corrupt_member is None, (
        f"Corrupt ZIP member: "
        f"{corrupt_member}"
    )

    zip_member_count = len(
        zip_file.namelist()
    )

    assert zip_member_count > 0


# ------------------------------------------------------------
# Publish ZIP atomically
# ------------------------------------------------------------
os.replace(
    TEMP_ZIP_PATH,
    ZIP_PATH
)

assert ZIP_PATH.exists()
assert ZIP_PATH.stat().st_size > 0


# ------------------------------------------------------------
# ZIP checksum
# ------------------------------------------------------------
zip_sha256 = calculate_sha256(
    ZIP_PATH
)

atomic_write_text(
    ZIP_SHA256_PATH,
    (
        f"{zip_sha256}  "
        f"{ZIP_PATH.name}\n"
    )
)

zip_size_mb = (
    ZIP_PATH.stat().st_size /
    (1024 ** 2)
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------
print("\n" + "=" * 100)
print("DENSENET121 EXPERIMENT — FINAL AUDIT COMPLETE")
print("=" * 100)

print("Run ID            :", RUN_ID)
print("Run folder        :", RUN_ROOT)
print("Best model        :", BEST_MODEL_PATH)
print("Final audit       :", FINAL_AUDIT_PATH)
print("Artifact manifest :", ARTIFACT_MANIFEST_PATH)
print("ZIP archive       :", ZIP_PATH)
print(
    "ZIP size          :",
    f"{zip_size_mb:.2f} MB"
)
print("ZIP members       :", zip_member_count)
print("ZIP SHA-256       :", zip_sha256)

print("\nFinal image-level metrics:")

print(
    saved_image_metrics.to_string(
        index=False
    )
)

print("=" * 100)
print("REQUIRED FILE AUDIT       : PASS")
print("TABLE QUALITY GATES       : PASS")
print("CLEAN-LOAD INFERENCE      : PASS")
print("FIGURE QUALITY GATES      : PASS")
print("ARTIFACT CHECKSUMS        : SAVED")
print("ZIP INTEGRITY TEST        : PASS")
print("EXPERIMENT STATUS         : COMPLETED")
print("CELL 15 COMPLETED")
print("=" * 100)



FINAL AUDIT, CLEAN-LOAD INFERENCE TEST AND ZIP EXPORT

Required directories:
----------------------------------------------------------------------------------------------------
checkpoints    : FOUND
logs           : FOUND
metrics        : FOUND
predictions    : FOUND
figures        : FOUND
metadata       : FOUND

Required files:
----------------------------------------------------------------------------------------------------
resolved_config             : FOUND
best_model                  : FOUND
last_model                  : FOUND
model_manifest              : FOUND
leakage_report              : FOUND
accounting_report           : FOUND
smoke_test_report           : FOUND
frozen_history              : FOUND
finetune_history            : FOUND
combined_history            : FOUND
image_metrics               : FOUND
source_metrics              : FOUND
classification_report       : FOUND
validation_threshold        : FOUND
final_evaluation            : FOUND
image_predictions        

In [25]:
# ============================================================
# FIGURE QUALITY CHECK
# ============================================================

from PIL import Image

figure_files = sorted(
    [p for p in FIGURE_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg"}],
    key=lambda p: p.name.lower(),
)

print("Figure folder:", FIGURE_DIR)
if not figure_files:
    print("No figure files found.")
else:
    for path in figure_files:
        with Image.open(path) as image:
            width, height = image.size
        assert min(width, height) >= 600, (
            f"Figure resolution is below the 600 px minimum: {path.name} -> {(width, height)}"
        )
        print(path.name, "|", (width, height), "| PASS")


Figure folder: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42/figures
test_confusion_matrices.png | (1656, 802) | PASS
test_probability_distribution.png | (1476, 876) | PASS
test_roc_pr_curves.png | (1776, 726) | PASS
training_validation_curves.png | (1626, 2076) | PASS


In [26]:
# ============================================================
# FINAL NOTEBOOK STATE CHECK
# ============================================================

assert RUN_ROOT.exists(), "Run root does not exist."
assert MANIFEST_PATH.exists(), "Model manifest does not exist."
assert LEAKAGE_REPORT_PATH.exists(), "Leakage report does not exist."

print("=" * 90)
print("DENSENET121 COMBINED EYE ROI NOTEBOOK: FINAL STATE PASS")
print("Run ID   :", RUN_ID)
print("Run root :", RUN_ROOT)
print("=" * 90)


DENSENET121 COMBINED EYE ROI NOTEBOOK: FINAL STATE PASS
Run ID   : 20260808_1214_eye_densenet121_seed42
Run root : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/DenseNet121_Eye_Results/20260808_1214_eye_densenet121_seed42
